# P05 추가 분석: 고친 GEO-SOM으로 원래 목표에 닿나 (회기동·동대문구, Colab)

README의 "추가 분석" 절(4~6단계)을 만든 코드입니다. `P05_repro_colab.ipynb`(결함 재현, 1~2단계)에 이어지는 분석이며, 처음부터 끝까지 한 번에 실행할 수 있게 준비 셀을 다시 넣었습니다.

- **질문:** 지리 성분을 고친 GEO-SOM 위에서 두 번째 군집을 어떻게 해야 "건물 물리 특성으로 도시를 다시 본다"는 목표에 닿는가?
- **좋은 군집의 정의 (두 축):** 속성 분리 = 건물 표준화 변수 5개의 실루엣 계수, 공간 응집 S = 가까운 이웃 5개가 같은 군집인 비율을 라벨을 99번 섞은 기준선으로 정규화한 값 (0 = 무작위, 1 = 이웃이 모두 같은 군집).
- **단순 기준선:** 속성만 쓴 K-Means(R1), 좌표만 쓴 K-Means(R2), 그리고 [속성, w × 좌표]로 w를 바꿔 가며 그린 절충 곡선(R3). GEO-SOM 후보는 이 곡선보다 위에 있어야 쓸 이유가 있다.
- **판정 기준:** 각 섹션 위에 적었습니다. 기준과 예측은 실행 전에 정했고, 결과를 본 뒤 바꾸지 않았습니다.
- **규모:** 회기동(건물 783개, 시드 5개)과 동대문구(속성 결측 제외 18,104개, 시드 1개). 서울 전체가 아닙니다. 이 레포의 GEO-SOM 구현에 대한 결과이며 GEO-SOM 방법 일반에 대한 결론이 아닙니다.
- **실행 시간 (Colab CPU):** 전체 약 40~60분. 가장 오래 걸리는 것은 Gap 수정판(약 10분)과 동대문구 SOM 학습(각 1~3분)입니다.
- **선택 입력:** 용도지역과의 비교에는 동대문구 용도지역 shapefile(`specialUse_dongDeamun.shp` 등, 국가공간정보포털 용도지역지구 자료에서 동대문구 부분)이 필요합니다. 레포에는 넣지 않았고, `/content` 아래에 올리면 사용하고 없으면 그 부분만 건너뜁니다.
- 그림은 `/content/figs_f/`, `/content/figs_f2/`에 저장됩니다(노트북에는 표시하지 않음). 레포 `figures/11~14`가 그중 일부입니다.

## 0. 준비 (P05_repro_colab.ipynb와 같은 셀)

데이터 zip 2개(회기동 경계 `data/hoegiDong.zip`, 동대문구 건물 `AL_D198_11230_...zip`)를 올리고, 원본 `utils.py`와 수정본 `GeoSOMFixed`를 정의합니다.

In [ ]:
import geopandas, sklearn, numpy, pandas, scipy
print('geopandas', geopandas.__version__, '| sklearn', sklearn.__version__, '| numpy', numpy.__version__, '| pandas', pandas.__version__)

In [ ]:
# 데이터 zip 2개를 올린다 (건물 zip 141MB는 시간이 좀 걸린다). 이미 /content에 있으면 건너뛴다.
import glob, os, zipfile
if not glob.glob('/content/AL_D198*.zip') or not glob.glob('/content/hoegiDong*.zip'):
    from google.colab import files
    files.upload()
for z in glob.glob('/content/AL_D198*.zip') + glob.glob('/content/hoegiDong*.zip'):
    zipfile.ZipFile(z).extractall('/content/data')
print(glob.glob('/content/data/**/*.shp', recursive=True))

In [ ]:
%%writefile utils.py
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy.spatial import distance_matrix
from shapely.geometry import Polygon, box, LineString
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon, MultiPoint
from scipy.spatial import Voronoi
from tqdm import tqdm  # tqdm 추가
import os


import time
import hashlib
import scipy
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, Birch
from sklearn.datasets import make_blobs


def construct_voronoi_polygons(X_geo, extend_distance=None):
    vor = Voronoi(X_geo)

    center = vor.points.mean(axis=0)
    if extend_distance is None:
        extend_distance = np.ptp(vor.points, axis=0).max() * 2 # 데이터 범위의 2배로 연장

    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))

    polygons = []

    for p_idx, region_idx in enumerate(vor.point_region):
        vertices = vor.regions[region_idx]
        if -1 not in vertices:
            # 유한한 영역 -> 바로 Polygon 생성
            poly_points = [vor.vertices[v] for v in vertices]
            polygons.append(Polygon(poly_points))
        else:
            # 무한한 영역 -> 보완이 필요
            ridge_segments = []
            ridges = all_ridges[p_idx]
            for p2, v1, v2 in ridges:
                v1, v2 = np.asarray(v1), np.asarray(v2)
                if v1 >= 0 and v2 >= 0:
                    # 유한한 에지
                    ridge_segments.append(vor.vertices[[v1, v2]])
                else:
                    # 무한한 에지
                    if v1 == -1:
                        v = v2
                    else:
                        v = v1
                    tangent = vor.points[p2] - vor.points[p_idx]
                    tangent = tangent / np.linalg.norm(tangent)
                    normal = np.array([-tangent[1], tangent[0]])

                    if np.dot(normal, vor.vertices[v] - center) < 0:
                        normal = -normal

                    far_point = vor.vertices[v] + normal * extend_distance
                    ridge_segments.append(np.vstack([vor.vertices[v], far_point]))

            # 이 모든 선분들의 꼭짓점을 연결
            poly_points = []
            for seg in ridge_segments:
                poly_points.append(seg[0])
                poly_points.append(seg[1])
            poly = Polygon(poly_points)
            polygons.append(poly.convex_hull)  # Convex Hull로 정리해서 Polygon 생성

    return polygons, vor


# ====== Geo-SOM Class ======
class GeoSOM:
    def __init__(self, grid_shape, geo_tolerance, input_dim, learning_rate=0.01, radius=2, fixed_geo=False):
        self.grid_shape = grid_shape
        self.geo_tolerance = geo_tolerance
        self.input_dim = input_dim  # X_ngf.shape[1] 대신 직접 받도록 수정
        self.learning_rate = learning_rate
        self.radius = radius
        self.fixed_geo = fixed_geo
        self.init_map()

    def init_map(self):
        rows, cols = self.grid_shape
        self.map_geo = np.array([[(i, j) for j in range(cols)] for i in range(rows)], dtype=np.float64).reshape(-1, 2)
        self.map_ngf = np.random.rand(rows * cols, self.input_dim)

    def train(self, X_geo, X_ngf, epochs=100):
        for epoch in tqdm(range(epochs), desc="Training GeoSOM"):  # tqdm 적용
            alpha = self.learning_rate * (1 - epoch / epochs)
            radius = self.radius * (1 - epoch / epochs)
            for i in range(len(X_geo)):
                x_geo = X_geo[i]
                x_ngf = X_ngf[i]
                dists_geo = np.linalg.norm(self.map_geo - x_geo, axis=1)
                geo_winner_idx = np.argmin(dists_geo)
                geo_winner_pos = self.map_geo[geo_winner_idx]
                neighborhood_idx = [j for j, unit_pos in enumerate(self.map_geo)
                                    if np.linalg.norm(geo_winner_pos - unit_pos) <= self.geo_tolerance]

                bmu_idx = min(neighborhood_idx,
                              key=lambda j: np.linalg.norm(x_ngf - self.map_ngf[j]))

                for j in neighborhood_idx:
                    dist_output = np.linalg.norm(self.map_geo[bmu_idx] - self.map_geo[j])
                    if dist_output <= radius:
                        h = np.exp(-(dist_output ** 2) / (2 * (radius ** 2)))
                        if self.fixed_geo:
                            self.map_ngf[j] += alpha * h * (x_ngf - self.map_ngf[j])
                        else:
                            combined_input = np.concatenate([x_geo, x_ngf])
                            combined_weight = np.concatenate([self.map_geo[j], self.map_ngf[j]])
                            delta = alpha * h * (combined_input - combined_weight)
                            self.map_geo[j] += delta[:2]
                            self.map_ngf[j] += delta[2:]


    def get_cluster_assignments(self, X_geo, X_ngf):
        assignments = []
        for i in range(len(X_geo)):
            x_geo = X_geo[i]
            x_ngf = X_ngf[i]
            dists_geo = np.linalg.norm(self.map_geo - x_geo, axis=1)
            geo_winner_idx = np.argmin(dists_geo)
            geo_winner_pos = self.map_geo[geo_winner_idx]
            neighborhood_idx = [j for j, unit_pos in enumerate(self.map_geo)
                                if np.linalg.norm(geo_winner_pos - unit_pos) <= self.geo_tolerance]
            bmu_idx = min(neighborhood_idx, key=lambda j: np.linalg.norm(x_ngf - self.map_ngf[j]))
            assignments.append(bmu_idx)
        return np.array(assignments)


# https://www.kaggle.com/code/mallikarjunaj/gap-statistics

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

def optimalK(data, maxClusters):
    """
    Calculates KMeans optimal K using Gap Statistic from Tibshirani, Walther, Hastie

    Params:
        data: ndarray of shape (n_samples, n_features)
        maxClusters: Maximum number of clusters to test for

    Returns: (optimal_k, resultsdf)
    """
    nrefs = 3
    gaps = np.zeros((len(range(1, maxClusters)),))
    resultsdf = pd.DataFrame(columns=['clusterCount', 'gap'])

    for gap_index, k in enumerate(range(1, maxClusters)):
        # 기준선 분산값 저장
        refDisps = np.zeros(nrefs)

        # nrefs개 기준선 데이터에 대해 KMeans
        for i in range(nrefs):
            randomReference = np.random.random_sample(size=data.shape)
            km = KMeans(n_clusters=k, n_init=10)
            km.fit(randomReference)
            refDisps[i] = km.inertia_

        # 실제 데이터에 대해 KMeans
        km = KMeans(n_clusters=k, n_init=10)
        km.fit(data)
        origDisp = km.inertia_

        # Gap 통계량 계산
        gap = np.log(np.mean(refDisps)) - np.log(origDisp)
        gaps[gap_index] = gap

        # 결과 누적
        new_row = pd.DataFrame([{'clusterCount': k, 'gap': gap}])
        resultsdf = pd.concat([resultsdf, new_row], ignore_index=True)

    optimal_k = gaps.argmax() + 1  # index 0 → k=1
    return optimal_k, resultsdf

In [ ]:
import sys
sys.path.insert(0, '/content')
from utils import *   # 원본 utils.py 그대로: GeoSOM, optimalK, construct_voronoi_polygons

In [ ]:
# --- 원 노트북 셀 7~19: 데이터 불러오기, 좌표계 맞추기, 회기동 clip, centroid ---
dongDea_gdf = gpd.read_file('/content/data/AL_D198_11230_20250115/AL_D198_11230_20250115.shp', encoding='euc-kr')
hoeGi_gdf = gpd.read_file('/content/data/hoegiDong/hoegi.shp', encoding='euc-kr')
dongDea_gdf = dongDea_gdf.to_crs('epsg:5179')
hoeGi_buildings = gpd.clip(dongDea_gdf, hoeGi_gdf)
hoeGi_buildings_centroid = hoeGi_buildings.copy()
hoeGi_buildings_centroid['geometry'] = hoeGi_buildings_centroid.geometry.centroid
print('동대문구', dongDea_gdf.shape, '| 회기동 건물', hoeGi_buildings.shape)

In [ ]:
# --- 원 노트북 셀 22~29: 전처리 (MinMax 뒤에 Standard를 연속 적용한 것까지 원본 그대로) ---
hoeGi_point = hoeGi_buildings_centroid.loc[:,['A20','A21','A30','A31','A32','A34','geometry']]
hoeGi_point['total_floor'] = hoeGi_point['A31'] + hoeGi_point['A32']
hoeGi_point = hoeGi_point.loc[:,['A20','A21','A30','A34','total_floor','geometry']]
hoeGi_point.dropna(inplace=True)
for idx, row in hoeGi_point.iterrows():
    x = int(row.values[3][:4])
    if x < 1970:
        hoeGi_point.at[idx,'A34'] = '1970-01-01'
hoeGi_point.dropna(inplace=True)
hoeGi_point['A34'] = pd.to_datetime(hoeGi_point['A34'])
reference_date = pd.to_datetime('2025-01-01')
hoeGi_point['years_old'] = (reference_date - hoeGi_point['A34']).dt.days // 365
hoeGi_point.drop(columns=['A34'], inplace=True)
from sklearn.preprocessing import MinMaxScaler, StandardScaler
cols = ['A20','A21','A30','total_floor','years_old']
hoeGi_point[cols] = MinMaxScaler().fit_transform(hoeGi_point[cols])
hoeGi_point[cols] = StandardScaler().fit_transform(hoeGi_point[cols])

X_geo = np.array(hoeGi_point.geometry.apply(lambda g: [g.x, g.y]).tolist())
X_ngf = hoeGi_point[cols].values
N = len(X_geo)
print('N =', N, '| X_geo 범위(m):', X_geo.min(0), X_geo.max(0), '| X_ngf', X_ngf.shape)

In [ ]:
from sklearn.neighbors import NearestNeighbors

class GeoSOMFixed:
    def __init__(self, grid_shape, geo_tolerance, input_dim, X_geo, learning_rate=0.01, radius=2, fixed_geo=False):
        self.grid_shape = grid_shape
        self.geo_tolerance = geo_tolerance
        self.input_dim = input_dim
        self.learning_rate = learning_rate
        self.radius = radius
        self.fixed_geo = fixed_geo
        rows, cols = grid_shape
        self.grid_pos = np.array([[(i, j) for j in range(cols)] for i in range(rows)], dtype=np.float64).reshape(-1, 2)
        xs = np.linspace(X_geo[:,0].min(), X_geo[:,0].max(), cols)
        ys = np.linspace(X_geo[:,1].min(), X_geo[:,1].max(), rows)
        self.map_geo = np.array([[(xs[j], ys[i]) for j in range(cols)] for i in range(rows)], dtype=np.float64).reshape(-1, 2)
        self.map_ngf = np.random.rand(rows * cols, input_dim)

    def _neigh(self, x_geo):
        w = np.argmin(np.linalg.norm(self.map_geo - x_geo, axis=1))
        d = np.linalg.norm(self.grid_pos - self.grid_pos[w], axis=1)   # tolerance는 격자 거리
        return w, np.where(d <= self.geo_tolerance)[0]

    def train(self, X_geo, X_ngf, epochs=100):
        for epoch in tqdm(range(epochs), desc="Training GeoSOMFixed"):
            alpha = self.learning_rate * (1 - epoch / epochs)
            radius = self.radius * (1 - epoch / epochs)
            for i in range(len(X_geo)):
                x_geo, x_ngf = X_geo[i], X_ngf[i]
                _, nb = self._neigh(x_geo)
                bmu_idx = nb[np.argmin(np.linalg.norm(self.map_ngf[nb] - x_ngf, axis=1))]
                for j in nb:
                    dist_output = np.linalg.norm(self.grid_pos[bmu_idx] - self.grid_pos[j])  # 반경도 격자 거리
                    if dist_output <= radius:
                        h = np.exp(-(dist_output ** 2) / (2 * (radius ** 2)))
                        self.map_ngf[j] += alpha * h * (x_ngf - self.map_ngf[j])
                        if not self.fixed_geo:
                            self.map_geo[j] += alpha * h * (x_geo - self.map_geo[j])

    def get_cluster_assignments(self, X_geo, X_ngf):
        out = []
        for i in range(len(X_geo)):
            _, nb = self._neigh(X_geo[i])
            out.append(nb[np.argmin(np.linalg.norm(self.map_ngf[nb] - X_ngf[i], axis=1))])
        return np.array(out)

def geo_winners(model, X_geo):
    return np.array([np.argmin(np.linalg.norm(model.map_geo - x, axis=1)) for x in X_geo])

def neigh_size_mean(model, winners):
    ref = model.grid_pos if hasattr(model, 'grid_pos') else model.map_geo   # 원본은 map_geo 거리 (원 코드 그대로)
    cache = {}
    for w in set(winners.tolist()):
        cache[w] = int((np.linalg.norm(ref - ref[w], axis=1) <= model.geo_tolerance).sum())
    return float(np.mean([cache[w] for w in winners]))

_nn_idx = NearestNeighbors(n_neighbors=6).fit(X_geo).kneighbors(X_geo)[1][:, 1:]
def cohesion_lift(labels, n_shuffle=20, seed=123):
    obs = (labels[_nn_idx] == labels[:, None]).mean()
    rng = np.random.default_rng(seed)
    base = np.mean([(rng.permutation(labels)[_nn_idx] == labels[:, None]).mean() for _ in range(n_shuffle)])
    return float(obs), float(base), float(obs / base)

RUNS = {}
def run_one(kind, fixed_geo, seed, epochs=50):
    np.random.seed(seed)
    if kind == 'orig':
        m = GeoSOM(grid_shape=(20, 20), geo_tolerance=4, input_dim=X_ngf.shape[1], fixed_geo=fixed_geo)
    else:
        m = GeoSOMFixed((20, 20), 4, X_ngf.shape[1], X_geo, fixed_geo=fixed_geo)
    m.train(X_geo, X_ngf, epochs=epochs)
    asg = m.get_cluster_assignments(X_geo, X_ngf)
    w = geo_winners(m, X_geo)
    vals, cnt = np.unique(w, return_counts=True)
    obs, base, lift = cohesion_lift(asg)
    r = dict(kind=kind, fixed_geo=fixed_geo, seed=seed, N=int(N),
             D_geo=int(len(vals)), top1=float(cnt.max() / N), top1_node=int(vals[cnt.argmax()]),
             D_bmu=int(len(np.unique(asg))), neigh_mean=neigh_size_mean(m, w),
             knn_obs=obs, knn_base=base, lift=lift)
    RUNS[(kind, fixed_geo, seed)] = dict(r=r, asg=asg, winners=w, model=m)
    return r

# ---- 기준 판정 (실행 전에 정한 기준 그대로) ----
def verdict_a(r):          # 원본용 (1장). (c)의 fixed_geo=False 원본에도 같은 표 적용
    if r['D_geo'] <= 2 or r['top1'] >= 0.90: return '결함 재현'
    if r['D_geo'] >= 20 and r['top1'] < 0.30: return '결함 아님(반증)'
    return '판정 보류'
def verdict_b(r, lift_a):  # 수정본용 (2장)
    ok = r['D_geo'] >= 100 and r['top1'] <= 0.10 and r['lift'] >= 2.0 and r['lift'] > lift_a
    return '수정 성공' if ok else '수정본 점검 필요(결론 보류)'

def show(keys):
    rows = []
    for k in keys:
        r = dict(RUNS[k]['r'])
        if r['kind'] == 'orig': r['판정'] = verdict_a(r)
        else:
            la = RUNS.get(('orig', True, r['seed']), {}).get('r', {}).get('lift', float('nan'))
            r['판정'] = verdict_b(r, la)
        rows.append(r)
    df = pd.DataFrame(rows)[['kind','fixed_geo','seed','N','D_geo','top1','top1_node','D_bmu','neigh_mean','knn_obs','knn_base','lift','판정']]
    display(df.round(3)); return df

def plot_run(key):
    d = RUNS[key]; r = d['r']
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    vals, cnt = np.unique(d['winners'], return_counts=True)
    ax[0].bar(vals, cnt); ax[0].set_xlim(-5, 405)
    ax[0].set_title(f"지리 승자 노드 분포 {key}  D_geo={r['D_geo']}, top1={r['top1']:.2f}")
    ax[0].set_xlabel('node index (0..399)'); ax[0].set_ylabel('건물 수')
    ax[1].scatter(X_geo[:,0], X_geo[:,1], c=d['asg'] % 20, cmap='tab20', s=8)
    ax[1].set_title(f"최종 군집 지도 D_bmu={r['D_bmu']}, lift={r['lift']:.2f}"); ax[1].set_aspect('equal')
    plt.show()

한글 그림 제목이 필요하면 아래 셀의 주석을 풀어 실행한 뒤 **런타임을 다시 시작**하고 처음부터 실행하세요. 없어도 계산에는 영향이 없습니다.

In [ ]:
# !apt-get -qq install -y fonts-nanum > /dev/null && ls /usr/share/fonts/truetype/nanum | head

## 1. 공통 함수

- `GeoSOMFixedV`: 수정본 `GeoSOMFixed`의 이웃 갱신만 벡터화한 것(계산 의미 동일). 다음 셀에서 원래 구현과 결과가 같은지 먼저 확인합니다.
- k 집합 {4, 6, 8, 10, 14}, 시드 0~4, SOM 20×20, epochs 50, tol=4가 기본값입니다.

In [ ]:
import numpy as np, pandas as pd, time, json, warnings, itertools, scipy.stats
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, glob
warnings.filterwarnings('ignore')
_f = glob.glob('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
if _f:
    fm.fontManager.addfont(_f[0]); plt.rcParams['font.family'] = 'NanumGothic'; plt.rcParams['axes.unicode_minus'] = False
print('font:', _f)

KS = [4, 6, 8, 10, 14]; K_MAIN = 8; SEEDS = [0, 1, 2, 3, 4]
WS = [0, 0.25, 0.5, 1, 2, 4, 8, 16]
TOLS = [0, 2, 4, 8, 30]          # 30 = 격자 대각선(26.9)보다 큼 = 지리 제약 없음
ROWS, LAB, SOMS = [], {}, {}      # ROWS: 지표 행, LAB[(scale,cand,param,k,seed)] = 건물 라벨, SOMS[(scale,tol,seed,grid)] = (model, bmu)

class GeoSOMFixedV(GeoSOMFixed):
    """GeoSOMFixed와 계산 의미가 같고 nb 갱신만 벡터화 (노드별 갱신은 서로 독립)."""
    def train(self, X_geo, X_ngf, epochs=100):
        for epoch in range(epochs):
            alpha = self.learning_rate * (1 - epoch / epochs)
            radius = self.radius * (1 - epoch / epochs)
            for i in range(len(X_geo)):
                x_geo, x_ngf = X_geo[i], X_ngf[i]
                _, nb = self._neigh(x_geo)
                bmu_idx = nb[np.argmin(np.linalg.norm(self.map_ngf[nb] - x_ngf, axis=1))]
                d = np.linalg.norm(self.grid_pos[nb] - self.grid_pos[bmu_idx], axis=1)
                m = d <= radius
                if m.any():
                    sel = nb[m]
                    h = np.exp(-(d[m] ** 2) / (2 * (radius ** 2)))[:, None]
                    self.map_ngf[sel] += alpha * h * (x_ngf - self.map_ngf[sel])
                    if not self.fixed_geo:
                        self.map_geo[sel] += alpha * h * (x_geo - self.map_geo[sel])

def train_som(Xg, Xn, tol, seed, grid=(20, 20), epochs=50, cls=None):
    cls = cls or GeoSOMFixedV
    np.random.seed(seed)
    m = cls(grid, tol, Xn.shape[1], Xg, fixed_geo=True)
    m.train(Xg, Xn, epochs=epochs)
    return m, m.get_cluster_assignments(Xg, Xn)

def make_nn(Xg): return NearestNeighbors(n_neighbors=6).fit(Xg).kneighbors(Xg)[1][:, 1:]

def spatial_S(labels, nn, n_shuffle=99, seed=123):
    obs = (labels[nn] == labels[:, None]).mean()
    rng = np.random.default_rng(seed)
    bs = []
    for _ in range(n_shuffle):
        lp = rng.permutation(labels); bs.append((lp[nn] == lp[:, None]).mean())
    base = float(np.mean(bs))
    return float(obs), base, float((obs - base) / (1 - base)) if base < 1 else float('nan')

def attr_metrics(Xn, labels, sample=None):
    ku = np.unique(labels)
    if len(ku) < 2: return dict(sil=float('nan'), r2=0.0, db=float('nan'), k_eff=int(len(ku)))
    if sample and len(Xn) > sample:
        idx = np.random.default_rng(0).choice(len(Xn), sample, replace=False)
        sil = silhouette_score(Xn[idx], labels[idx])
    else:
        sil = silhouette_score(Xn, labels)
    tot = ((Xn - Xn.mean(0)) ** 2).sum()
    within = sum(((Xn[labels == c] - Xn[labels == c].mean(0)) ** 2).sum() for c in ku)
    return dict(sil=float(sil), r2=float(1 - within / tot), db=float(davies_bouldin_score(Xn, labels)), k_eff=int(len(ku)))

def record(scale, cand, param, k, seed, labels, Xn, nn, sample=None):
    labels = np.asarray(labels)
    obs, base, S = spatial_S(labels, nn)
    r = dict(scale=scale, cand=cand, param=param, k=k, seed=seed, **attr_metrics(Xn, labels, sample),
             obs=obs, base=base, S=S, lift=obs / base)
    ROWS.append(r); LAB[(scale, cand, param, k, seed)] = labels
    return r

def km(X, k, seed): return KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(X)

def coords_std(Xg):
    c = Xg - Xg.mean(0); return c / np.sqrt(c.var(0).mean())   # x·y 공통 척도

def node_labels_km(W, k, seed): return KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(W)
def node_labels_ward(W, k): return AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(W)

def cluster_mean_xy(Xg, grp):
    out = np.zeros_like(Xg)
    for g in np.unique(grp):
        m = grp == g; out[m] = Xg[m].mean(0)
    return out

def run_baselines(scale, Xg, Xn, nn, seeds, sample=None):
    Z = coords_std(Xg)
    for k in KS:
        for s in seeds:
            record(scale, 'R1', 0, k, s, km(Xn, k, s), Xn, nn, sample)
            record(scale, 'R2', 0, k, s, km(Z, k, s), Xn, nn, sample)
            for w in WS:
                X3 = Xn if w == 0 else np.hstack([Xn, w * Z])
                record(scale, 'R3', w, k, s, km(X3, k, s), Xn, nn, sample)

def run_som_candidates(scale, Xg, Xn, nn, seeds, tols, grid=(20, 20), sample=None, do_A=True, do_C=True):
    for tol in tols:
        for s in seeds:
            t0 = time.time()
            m, bmu = train_som(Xg, Xn, tol, s, grid=grid)
            SOMS[(scale, tol, s, grid)] = (m, bmu)
            W = m.map_ngf
            for k in KS:
                record(scale, 'B', tol, k, s, node_labels_km(W, k, s)[bmu], Xn, nn, sample)
                if do_C and tol == 4:
                    record(scale, 'C', tol, k, s, node_labels_ward(W, k)[bmu], Xn, nn, sample)
                if do_A and tol == 4:
                    XA = np.hstack([Xg, cluster_mean_xy(Xg, bmu)])
                    record(scale, 'A_fix', tol, k, s, km(XA, k, s), Xn, nn, sample)
            print(f'{scale} tol={tol} seed={s} grid={grid}: {time.time()-t0:.1f}s, D_bmu={len(np.unique(bmu))}')

def df_rows(): return pd.DataFrame(ROWS)

def med(scale, cand, param, k, key):
    v = [r[key] for r in ROWS if r['scale'] == scale and r['cand'] == cand and r['param'] == param and r['k'] == k]
    return (float(np.median(v)), float(np.max(v) - np.min(v)), len(v)) if v else (float('nan'), float('nan'), 0)

def delta(scale, cand, param, k, curve_scale=None, eps_fixed=None):
    cs = curve_scale or scale
    pts = sorted((med(cs, 'R3', w, k, 'S')[0], med(cs, 'R3', w, k, 'sil')[0]) for w in WS)
    xs, ys = np.array([p[0] for p in pts]), np.array([p[1] for p in pts])
    S_b = med(scale, cand, param, k, 'S')[0]; sil_b, rng_b, _ = med(scale, cand, param, k, 'sil')
    eps = eps_fixed if eps_fixed is not None else max(0.02, rng_b)
    if not (xs.min() <= S_b <= xs.max()): return dict(k=k, S=S_b, sil=sil_b, curve=None, delta=None, eps=eps, cls='비교 불가')
    cv = float(np.interp(S_b, xs, ys)); d = sil_b - cv
    return dict(k=k, S=S_b, sil=sil_b, curve=cv, delta=d, eps=eps, cls='+' if d > eps else ('-' if d < -eps else '0'))

def ari_within(scale, cand, param, k):
    labs = [LAB[(scale, cand, param, k, s)] for s in SEEDS if (scale, cand, param, k, s) in LAB]
    v = [adjusted_rand_score(a, b) for a, b in itertools.combinations(labs, 2)]
    return float(np.median(v)) if v else float('nan')

def ari_between(scale, c1, p1, c2, p2, k):
    v = [adjusted_rand_score(LAB[(scale, c1, p1, k, s)], LAB[(scale, c2, p2, k, s)])
         for s in SEEDS if (scale, c1, p1, k, s) in LAB and (scale, c2, p2, k, s) in LAB]
    return float(np.median(v)) if v else float('nan')

def stab_name(a): return '안정' if a >= 0.75 else ('보통' if a >= 0.5 else '불안정')

def verdict1(scale, cand='B', param=4, curve_scale=None):
    ds = [delta(scale, cand, param, k, curve_scale) for k in KS]
    up = sum(d['cls'] == '+' for d in ds); dn = sum(d['cls'] == '-' for d in ds)
    stab = float(np.median([ari_within(scale, cand, param, k) for k in KS]))
    if up >= 3 and stab_name(stab) != '불안정': v = 'GEO-SOM 우위'
    elif dn >= 3: v = 'GEO-SOM 열위'
    else: v = '구분 불가'
    return dict(verdict=v, up=up, down=dn, stab_ari=stab, stab=stab_name(stab), per_k=ds)

def verdict2(scale, cand):
    p = 4 if cand == 'A_fix' else 0
    a = float(np.median([ari_between(scale, cand, p, 'R2', 0, k) for k in KS]))
    low = sum(med(scale, cand, p, k, 'sil')[0] <= (med(scale, 'R1', 0, k, 'sil')[0] + med(scale, 'R2', 0, k, 'sil')[0]) / 2 for k in KS)
    if a >= 0.5 and low >= 3: v = '사실상 좌표 군집'
    elif (5 - low) >= 3: v = '속성 일부 반영'
    else: v = '보류'
    return dict(verdict=v, ari_R2=a, n_k_sil_below_mid=int(low))

def cramers_v(a, b):
    ct = pd.crosstab(pd.Series(a), pd.Series(b)).values
    if min(ct.shape) < 2: return float('nan')
    chi2 = scipy.stats.chi2_contingency(ct, correction=False)[0]
    return float(np.sqrt(chi2 / (ct.sum() * (min(ct.shape) - 1))))

nn_h = make_nn(X_geo)
print('F1 준비 완료. N =', N, '| X_ngf', X_ngf.shape)

### 1-1. 벡터화 구현 점검

기준: 회기동 tol=4·시드 0에서 원래 `GeoSOMFixed`와 건물별 배정(BMU)이 783개 전부 같아야 사용합니다.

In [ ]:
t0 = time.time(); m_o, bmu_o = train_som(X_geo, X_ngf, 4, 0, cls=GeoSOMFixed); t_o = time.time() - t0
t0 = time.time(); m_v, bmu_v = train_som(X_geo, X_ngf, 4, 0, cls=GeoSOMFixedV); t_v = time.time() - t0
CHECK_V = dict(all_equal=bool((bmu_o == bmu_v).all()), n_diff=int((bmu_o != bmu_v).sum()),
               D_bmu_orig=int(len(np.unique(bmu_o))), D_bmu_vec=int(len(np.unique(bmu_v))),
               max_abs_w_diff=float(np.abs(m_o.map_ngf - m_v.map_ngf).max()), t_orig=round(t_o, 1), t_vec=round(t_v, 1))
print(CHECK_V, '->', '통과: GeoSOMFixedV 사용' if CHECK_V['all_equal'] else '불일치: 원래 구현으로 돌림')

## 2. 회기동: 기준선과 후보 실행 (4단계)

- R1 속성만 / R2 좌표만 / R3(w) 속성+좌표 가중 K-Means(w = 0~16)
- B(tol): 수정 GEO-SOM 노드 가중치(400개)를 K-Means로 묶고 건물은 자기 노드의 군집을 따름. tol = 0(지리 칸 그대로), 2, 4, 8, 30(지리 제약 없음)
- C(4): 같은 노드 가중치를 Ward로 묶음
- A_fix / A_orig: 원 설계 [x, y, x′, y′] K-Means. x′, y′는 같은 GEO-SOM 군집 건물의 평균 좌표(보로노이 대신 건물 좌표로 단순화). A_orig는 원본 GEO-SOM 결과 위에서

In [ ]:
t0 = time.time()
run_baselines('hoegi', X_geo, X_ngf, nn_h, SEEDS)
print(f'기준선 {time.time()-t0:.1f}s')
run_som_candidates('hoegi', X_geo, X_ngf, nn_h, SEEDS, TOLS)
# A_orig: 원본 GeoSOM (a) 시드 0 (RUNS에 있으면 재사용)
if 'RUNS' in globals() and ('orig', True, 0) in RUNS:
    asg_orig = RUNS[('orig', True, 0)]['asg']
else:
    np.random.seed(0); _m = GeoSOM(grid_shape=(20, 20), geo_tolerance=4, input_dim=X_ngf.shape[1], fixed_geo=True)
    _m.train(X_geo, X_ngf, epochs=50); asg_orig = _m.get_cluster_assignments(X_geo, X_ngf)
print('A_orig 원본 군집 수 =', len(np.unique(asg_orig)))
XA_orig = np.hstack([X_geo, cluster_mean_xy(X_geo, asg_orig)])
for k in KS:
    for s in SEEDS: record('hoegi', 'A_orig', 0, k, s, km(XA_orig, k, s), X_ngf, nn_h)
print(f'F3 완료 {time.time()-t0:.1f}s, 행 {len(ROWS)}')

### 2-1. 판정

- **판정 1 (B(4) vs 절충 곡선):** k마다 같은 공간 응집 S에서 곡선과의 실루엣 차이 Δ를 구하고, ε = max(0.02, B(4) 실루엣의 시드 간 범위)로 둡니다. 5개 k 중 3개 이상에서 Δ > ε이고 안정성(시드 간 ARI 중앙값)이 0.5 이상이면 "우위", Δ < −ε이면 "열위", 그 외는 "구분 불가". S가 곡선 범위 밖이면 그 k는 "비교 불가".
- **판정 2 (원 설계):** 좌표만 쓴 K-Means와의 ARI 중앙값 ≥ 0.5이고 5개 k 중 3개 이상에서 실루엣이 R1과 R2의 중간 이하이면 "사실상 좌표 군집".

In [ ]:
V1 = verdict1('hoegi')
V2 = {c: verdict2('hoegi', c) for c in ('A_fix', 'A_orig')}
print('판정 1 B(4):', V1['verdict'], '| +', V1['up'], '-', V1['down'], '| 안정성', round(V1['stab_ari'], 3), V1['stab'])
for d in V1['per_k']: print('   ', {kk: (round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in d.items()})
print('판정 2:', V2)
EXPL = {}
EXPL['delta_other'] = {f'B({t})': [delta('hoegi', 'B', t, k)['cls'] for k in KS] for t in TOLS}
EXPL['delta_other']['C(4)'] = [delta('hoegi', 'C', 4, k)['cls'] for k in KS]
def _inv(seq, inc):  # 순서 뒤집힘 수
    return sum((b < a) if inc else (b > a) for a, b in zip(seq, seq[1:]))
EXPL['tol_dial'] = {k: dict(S=[round(med('hoegi', 'B', t, k, 'S')[0], 3) for t in TOLS],
                            sil=[round(med('hoegi', 'B', t, k, 'sil')[0], 3) for t in TOLS]) for k in KS}
EXPL['tol_dial_ok'] = all(_inv(v['S'], False) <= 1 and _inv(v['sil'], True) <= 1 for v in EXPL['tol_dial'].values())
EXPL['ari_Binf_R1_kmain'] = ari_between('hoegi', 'B', 30, 'R1', 0, K_MAIN)
EXPL['ari_B4_C4'] = {k: ari_between('hoegi', 'B', 4, 'C', 4, k) for k in KS}
EXPL['S_R1_k8'] = med('hoegi', 'R1', 0, 8, 'S')[0]
EXPL['stab'] = {c: float(np.median([ari_within('hoegi', c, p, k) for k in KS]))
                for c, p in [('R1', 0), ('R2', 0), ('B', 4), ('C', 4), ('A_fix', 4)]}
print(json.dumps(EXPL, ensure_ascii=False, default=float))
tab = df_rows().query("scale=='hoegi'").groupby(['cand','param','k'])[['sil','S','r2','lift']].median().round(3).unstack('k')
print(tab[['sil','S']].to_string())

## 3. Gap 수정판 (군집 수 선택)

원래 `optimalK`는 기준 분포가 [0, 1] 균일분포, 기준 표본 3개, 단순 최댓값 선택입니다. 수정판은 데이터 주성분 범위 상자의 균일분포, 기준 표본 20개, 1-SE 규칙(Gap(k) ≥ Gap(k+1) − s(k+1)인 가장 작은 k)입니다.

기준: maxClusters 15와 30에서 각 5회, 모두 상한보다 작고 두 설정의 k 중앙값 차이 ≤ 1이면 "상한 추종 완화". 30에서 3회 이상 k ≥ 25이면 "상한 추종 남음".

In [ ]:
def gap_f(X, maxClusters, nrefs=20, seed=0):
    rng = np.random.default_rng(seed)
    Xc = X - X.mean(0); Vt = np.linalg.svd(Xc, full_matrices=False)[2]; Xp = Xc @ Vt.T
    lo, hi = Xp.min(0), Xp.max(0)
    gaps, sks = [], []
    for k in range(1, maxClusters):
        Wk = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(Xp).inertia_
        lr = np.array([np.log(KMeans(n_clusters=k, n_init=10, random_state=b).fit(rng.uniform(lo, hi, size=Xp.shape)).inertia_)
                       for b in range(nrefs)])
        gaps.append(lr.mean() - np.log(Wk)); sks.append(lr.std() * np.sqrt(1 + 1 / nrefs))
    for i in range(len(gaps) - 1):
        if gaps[i] >= gaps[i + 1] - sks[i + 1]: return i + 1, gaps, sks
    return len(gaps), gaps, sks

GAP = {}
for name, X in [('Xk_Aorig', XA_orig), ('R1_attr', X_ngf)]:
    GAP[name] = {}
    for mc in (15, 30):
        GAP[name][mc] = []
        for r in range(5):
            t0 = time.time(); k, g, sk = gap_f(X, mc, seed=r)
            GAP[name][mc].append(dict(k=int(k), gap=[float(x) for x in g], sk=[float(x) for x in sk]))
            print(f'{name} mc={mc} rep={r}: k={k} ({time.time()-t0:.0f}s)')
def verdict3(G):
    k15 = [d['k'] for d in G[15]]; k30 = [d['k'] for d in G[30]]
    if all(k < 14 for k in k15) and all(k < 29 for k in k30) and abs(np.median(k15) - np.median(k30)) <= 1: v = '상한 추종 완화'
    elif sum(k >= 25 for k in k30) >= 3: v = '상한 추종 남음'
    else: v = '보류'
    return dict(verdict=v, k15=k15, k30=k30)
V3 = {n: verdict3(G) for n, G in GAP.items()}
# 같은 입력의 실루엣 argmax·엘보(관찰)
SIL_ELBOW = {}
for name, X in [('Xk_Aorig', XA_orig), ('R1_attr', X_ngf)]:
    sils, inert = {}, {}
    for k in range(1, 30):
        mdl = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X); inert[k] = float(mdl.inertia_)
        if k >= 2: sils[k] = float(silhouette_score(X, mdl.labels_))
    SIL_ELBOW[name] = dict(sil_argmax=int(max(sils, key=sils.get)), sil=sils, inertia=inert)
print('판정 3:', V3, '| 실루엣 argmax:', {n: v['sil_argmax'] for n, v in SIL_ELBOW.items()})

## 4. SOM 품질과 격자 크기

양자화 오차(QE)와 위상 오차(TE, 속성 공간 1·2위 노드가 격자에서 8-이웃이 아닌 비율)를 기록하고, 12×12 격자에서 판정 1을 다시 합니다(판정 이름이 같으면 "격자 크기에 강건").

In [ ]:
def qe_te(m, Xn):
    D = np.linalg.norm(Xn[:, None, :] - m.map_ngf[None, :, :], axis=2)
    o = np.argsort(D, axis=1)[:, :2]
    qe = float(D[np.arange(len(Xn)), o[:, 0]].mean())
    gd = np.linalg.norm(m.grid_pos[o[:, 0]] - m.grid_pos[o[:, 1]], axis=1)
    return qe, float((gd > np.sqrt(2) + 1e-9).mean())   # 8-이웃이 아니면 위상 오차
S1 = {'qe_te': {}}
for tol in TOLS:
    v = [qe_te(SOMS[('hoegi', tol, s, (20, 20))][0], X_ngf) for s in SEEDS]
    S1['qe_te'][tol] = dict(qe=float(np.median([a for a, _ in v])), te=float(np.median([b for _, b in v])))
m100, _ = train_som(X_geo, X_ngf, 4, 0, epochs=100)
qe50 = qe_te(SOMS[('hoegi', 4, 0, (20, 20))][0], X_ngf)[0]; qe100 = qe_te(m100, X_ngf)[0]
S1['epochs'] = dict(qe50=qe50, qe100=qe100, rel_change=float(abs(qe100 - qe50) / qe50))
run_som_candidates('hoegi_g12', X_geo, X_ngf, nn_h, SEEDS, [4], grid=(12, 12), do_A=False, do_C=False)
V4 = verdict1('hoegi_g12', curve_scale='hoegi')
V4['same_as_V1'] = V4['verdict'] == V1['verdict']
S1['qe_te_g12'] = dict(zip(['qe', 'te'], np.median([qe_te(SOMS[('hoegi_g12', 4, s, (12, 12))][0], X_ngf) for s in SEEDS], axis=0).tolist()))
print('S1:', json.dumps(S1, default=float)); print('판정 4:', V4['verdict'], '->', '격자 크기에 강건' if V4['same_as_V1'] else '격자 크기 의존')
for d in V4['per_k']: print('   ', {kk: (round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in d.items()})
print('g12 안정성', round(V4['stab_ari'],3), V4['stab'])

## 5. 전처리 민감도

- MinMax 다음 Standard를 적용한 값과 Standard 단독 값의 최대 차이 < 1e-9이면 "두 번째 스케일러만 남음".
- 용적률·높이·총층수에 log1p를 적용한 뒤 판정 1을 다시 합니다.
- 변수별 0 값 개수, 상관, 변수 하나를 뺐을 때의 ARI를 기록합니다.

In [ ]:
H = hoeGi_buildings_centroid.loc[hoeGi_point.index]
raw = pd.DataFrame(index=hoeGi_point.index)
raw['A20'] = H['A20']; raw['A21'] = H['A21']; raw['A30'] = H['A30']; raw['total_floor'] = H['A31'] + H['A32']
yrs_h = pd.to_numeric(H['A34'].astype(str).str[:4], errors='coerce')
a34 = H['A34'].astype(str).where(yrs_h >= 1970, '1970-01-01')
raw['years_old'] = (pd.to_datetime('2025-01-01') - pd.to_datetime(a34)).dt.days // 365
raw = raw[cols].astype(float)
std_only = StandardScaler().fit_transform(raw.values)
mm_std = StandardScaler().fit_transform(MinMaxScaler().fit_transform(raw.values))
S3 = dict(raw_matches_X_ngf=bool(np.allclose(mm_std, X_ngf, atol=1e-9)),
          v5_max_abs_diff=float(np.abs(std_only - mm_std).max()),
          n_pre1970=int((yrs_h < 1970).sum()),
          zeros={c: int((raw[c] == 0).sum()) for c in cols},
          corr=raw.corr().round(3).to_dict())
S3['v5'] = '두 번째 스케일러만 남음 확인' if S3['v5_max_abs_diff'] < 1e-9 else '차이 있음'
print('판정 5:', S3['v5'], S3['v5_max_abs_diff'], '| raw 재구성 일치:', S3['raw_matches_X_ngf'])
# 판정 6: 로그 변환
rl = raw.copy()
for c in ['A20', 'A30', 'total_floor']: rl[c] = np.log1p(rl[c])
X_log = StandardScaler().fit_transform(rl.values)
run_baselines('hoegi_log', X_geo, X_log, nn_h, SEEDS)
run_som_candidates('hoegi_log', X_geo, X_log, nn_h, SEEDS, [4], do_A=False, do_C=False)
V6 = verdict1('hoegi_log'); V6['same_as_V1'] = V6['verdict'] == V1['verdict']
print('판정 6:', V6['verdict'], '->', '변환에 강건' if V6['same_as_V1'] else '변환에 따라 다름')
for d in V6['per_k']: print('   ', {kk: (round(vv, 4) if isinstance(vv, float) else vv) for kk, vv in d.items()})
# 관찰: 높이 또는 total_floor를 뺐을 때 k_main ARI (R1 시드 0, B(4) 시드 0)
S3['drop_var'] = {}
for drop in ['A30', 'total_floor']:
    keep = [i for i, c in enumerate(cols) if c != drop]; Xd = std_only[:, keep]
    r1 = adjusted_rand_score(LAB[('hoegi', 'R1', 0, K_MAIN, 0)], km(Xd, K_MAIN, 0))
    md, bd = train_som(X_geo, Xd, 4, 0)
    b4 = adjusted_rand_score(LAB[('hoegi', 'B', 4, K_MAIN, 0)], node_labels_km(md.map_ngf, K_MAIN, 0)[bd])
    S3['drop_var'][drop] = dict(R1=float(r1), B4=float(b4))
print('S3:', json.dumps(S3, ensure_ascii=False, default=float))
print(raw.describe().round(2).to_string())

## 6. 동대문구 확인 (시드 0)

회기동 판정 1의 k별 분류(+/0/−)와 동대문구 분류(ε 0.02 고정)가 5개 k 중 3개 이상 같으면 "구 규모에서도 같은 방향". 분류는 +, 0, − 세 가지만 세며 "비교 불가"끼리의 일치는 세지 않습니다(이 셀의 출력은 비교 불가 일치도 세므로 README는 기준 문장대로 적었습니다).

In [ ]:
G = dongDea_gdf.copy(); G['geometry'] = G.geometry.centroid
GP = G.loc[:, ['A20', 'A21', 'A30', 'A31', 'A32', 'A34', 'geometry']].copy()
GP['total_floor'] = GP['A31'] + GP['A32']
GP = GP.loc[:, ['A20', 'A21', 'A30', 'A34', 'total_floor', 'geometry']].dropna()
yrs_g = pd.to_numeric(GP['A34'].astype(str).str[:4], errors='coerce')
GP = GP[yrs_g.notna()].copy(); yrs_g = yrs_g[yrs_g.notna()]
GP.loc[yrs_g < 1970, 'A34'] = '1970-01-01'
GP['years_old'] = (pd.to_datetime('2025-01-01') - pd.to_datetime(GP['A34'])).dt.days // 365
G_raw = GP[cols].astype(float)
G_ngf = StandardScaler().fit_transform(MinMaxScaler().fit_transform(G_raw.values))
G_geo = np.array(GP.geometry.apply(lambda g: [g.x, g.y]).tolist()); GN = len(G_geo)
nn_g = make_nn(G_geo)
print('동대문구 N =', GN, '| 0 개수:', {c: int((G_raw[c] == 0).sum()) for c in cols})
t0 = time.time(); run_baselines('gu', G_geo, G_ngf, nn_g, [0], sample=5000); print(f'기준선 {time.time()-t0:.0f}s')
run_som_candidates('gu', G_geo, G_ngf, nn_g, [0], [0, 4, 30], sample=5000, do_C=False)
V7_rows = []
for k in KS:
    h = delta('hoegi', 'B', 4, k)['cls']; g = delta('gu', 'B', 4, k, eps_fixed=0.02)
    V7_rows.append(dict(k=k, hoegi=h, gu=g['cls'], gu_delta=g['delta'], gu_S=g['S'], gu_sil=g['sil']))
V7 = dict(per_k=V7_rows, same=sum(r['hoegi'] == r['gu'] for r in V7_rows))
V7['verdict'] = '구 규모에서도 같은 방향' if V7['same'] >= 3 else '규모에 따라 다름'
print('판정 7:', V7)
print(df_rows().query("scale=='gu'").groupby(['cand','param','k'])[['sil','S']].median().round(3).unstack('k').to_string())

## 7. 군집 프로파일과 외부 변수 (k=8)

- 군집 이름은 |z| ≥ 0.5인 변수와 부호로 붙입니다(예: `A20+, A30+` = 용적률·높이 높음).
- 같은 이름의 군집이 시드 5개 중 4개 이상에서 나오는지(헝가리안 매칭)를 봅니다.
- 주용도(A25), 법정동(A3)과의 Cramér's V를 기록합니다.

In [ ]:
def profile_name(z, thr=0.5):
    return ', '.join(f"{c}{'+' if z[c] > 0 else '-'}" for c in cols if abs(z[c]) >= thr) or '평균 근처'
def profiles(raw_df, Xn, labels):
    zdf = pd.DataFrame(Xn, columns=cols).groupby(labels).mean()
    rdf = raw_df.reset_index(drop=True).groupby(labels).mean(); n = pd.Series(labels).value_counts().sort_index()
    out = pd.DataFrame({'n': n, 'name': [profile_name(zdf.loc[i]) for i in zdf.index]}).join(rdf.round(2)).join(zdf.round(2), rsuffix='_z')
    out.index = out.index.astype(str); return out
def rep_seed(scale, cand, p, k):
    m_ = {s: np.mean([adjusted_rand_score(LAB[(scale, cand, p, k, s)], LAB[(scale, cand, p, k, t)]) for t in SEEDS if t != s]) for s in SEEDS}
    target = ari_within(scale, cand, p, k); return min(m_, key=lambda s: abs(m_[s] - target))
def recur(scale, cand, p, k, Xn):   # 인사이트 조건 1: 시드 0 군집 프로파일 이름이 헝가리안 매칭 상대와 같은 시드 수 (자기 포함)
    def cents(s): L = LAB[(scale, cand, p, k, s)]; u = np.unique(L); return u, np.array([Xn[L == c].mean(0) for c in u])
    u0, c0 = cents(0); names0 = [profile_name(pd.Series(c, index=cols)) for c in c0]; cnt = np.ones(len(u0), int)
    for s in SEEDS[1:]:
        u1, c1 = cents(s); C = np.linalg.norm(c0[:, None] - c1[None], axis=2); ri, ci = linear_sum_assignment(C)
        for i, j in zip(ri, ci):
            if profile_name(pd.Series(c1[j], index=cols)) == names0[i]: cnt[i] += 1
    return pd.DataFrame({'cluster': u0, 'name': names0, 'n_seeds_same': cnt})
S4 = {}
for cand, p in [('R1', 0), ('B', 4)]:
    rs = rep_seed('hoegi', cand, p, K_MAIN)
    S4[f'{cand}{p}'] = dict(rep_seed=int(rs),
        prof_seed0=profiles(raw, X_ngf, LAB[('hoegi', cand, p, K_MAIN, 0)]).to_dict(orient='index'),
        prof_rep=profiles(raw, X_ngf, LAB[('hoegi', cand, p, K_MAIN, rs)]).to_dict(orient='index'),
        recur=recur('hoegi', cand, p, K_MAIN, X_ngf).to_dict(orient='records'),
        gu_names=sorted(set(profiles(G_raw, G_ngf, LAB[('gu', cand, p, K_MAIN, 0)])['name'])))
# 주용도(A25)·법정동(A3) Cramér's V (관찰)
use_h = hoeGi_buildings_centroid.loc[hoeGi_point.index, 'A25'].astype(str).values
use_g = dongDea_gdf.loc[GP.index, 'A25'].astype(str).values; dong_g = dongDea_gdf.loc[GP.index, 'A3'].astype(str).values
S4['V_use'] = {c: dict(hoegi=float(np.median([cramers_v(LAB[('hoegi', c, p, K_MAIN, s)], use_h) for s in SEEDS])),
                       gu=cramers_v(LAB[('gu', c, p, K_MAIN, 0)], use_g)) for c, p in [('R1', 0), ('R2', 0), ('B', 4)]}
S4['V_dong_gu'] = {c: cramers_v(LAB[('gu', c, p, K_MAIN, 0)], dong_g) for c, p in [('R1', 0), ('R2', 0), ('B', 4)]}
S4['use_counts_hoegi'] = pd.Series(use_h).value_counts().to_dict()
print(json.dumps({k: S4[k] for k in ('V_use', 'V_dong_gu', 'use_counts_hoegi')}, ensure_ascii=False))
for k_ in ('R10', 'B4'): print(k_, 'rep_seed', S4[k_]['rep_seed'], 'recur:', S4[k_]['recur'], '| gu names:', S4[k_]['gu_names'])
pd.set_option('display.width', 250)
for k_ in ('R10', 'B4'): print(k_, 'seed0 profile'); print(pd.DataFrame(S4[k_]['prof_seed0']).T.to_string())
zero3 = (raw[['A20','A21','A30']] == 0).all(1).values
print('회기동 3변수 모두 0:', int(zero3.sum()), '| R1 k8 군집별 0묶음 비율:', pd.Series(zero3).groupby(LAB[('hoegi','R1',0,8,0)]).mean().round(2).to_dict())
print('B4 k8 군집별 0묶음 비율:', pd.Series(zero3).groupby(LAB[('hoegi','B',4,8,0)]).mean().round(2).to_dict())
print('0묶음 주용도:', pd.Series(use_h[zero3]).value_counts().head(8).to_dict(), '| 비0 주용도:', pd.Series(use_h[~zero3]).value_counts().head(8).to_dict())
print(sorted(glob.glob('/content/**/specialUse*', recursive=True)))

### 7-1. 용도지역 (선택)

용도지역 shapefile이 `/content` 아래에 있으면 건물 centroid로 공간 조인해 6분류(1종·2종·3종일반, 준주거, 상업, 자연녹지)로 묶고 Cramér's V를 냅니다. 읽는 규칙: V(속성 군집, 용도지역) ≥ 0.3이면 "속성 군집이 용도지역을 상당 부분 다시 찾아냄".

In [ ]:
import geopandas as gpd
_z = glob.glob('/content/**/specialUse_dongDeamun.shp', recursive=True)
HAS_ZONING = bool(_z)
if _z:
    Z = gpd.read_file(_z[0], encoding='cp949').to_crs('EPSG:5179')
    code_col = [c for c in Z.columns if str(c).startswith('용도지역')][0]
    grp = {'UQA121': '1종일반', 'UQA122': '2종일반', 'UQA120': '2종일반', 'UQA123': '3종일반', 'UQA130': '준주거',
           'UQA220': '상업', 'UQA230': '상업', 'UQA240': '상업', 'UQA430': '자연녹지'}
    Z['zone6'] = Z[code_col].map(grp)
    def zone_of(points_gdf):
        j = gpd.sjoin(points_gdf[['geometry']], Z[['zone6', 'geometry']], how='left', predicate='within')
        j = j[~j.index.duplicated(keep='first')]; return j['zone6'].reindex(points_gdf.index)
    zh = zone_of(hoeGi_point); zg = zone_of(GP)
    mh, mg = zh.notna().values, zg.notna().values
    S4['zoning'] = dict(code_col=code_col, n_unmatched_hoegi=int((~mh).sum()), n_unmatched_gu=int((~mg).sum()),
        counts_hoegi=zh.value_counts().to_dict(), counts_gu=zg.value_counts().to_dict(),
        V={c: dict(hoegi=float(np.median([cramers_v(LAB[('hoegi', c, p, K_MAIN, s)][mh], zh.values[mh]) for s in SEEDS])),
                   gu=cramers_v(LAB[('gu', c, p, K_MAIN, 0)][mg], zg.values[mg])) for c, p in [('R1', 0), ('R2', 0), ('B', 4)]})
    print('용도지역:', json.dumps(S4['zoning'], ensure_ascii=False))
    zero3g = (G_raw[['A20','A21','A30']] == 0).all(1).values
    S4['zoning']['zero3_share_by_zone_gu'] = pd.Series(zero3g[mg]).groupby(zg.values[mg]).mean().round(3).to_dict()
    S4['zoning']['zero3_share_by_zone_hoegi'] = pd.Series(zero3[mh]).groupby(zh.values[mh]).mean().round(3).to_dict()
    print('0묶음 비율(용도지역별) gu:', S4['zoning']['zero3_share_by_zone_gu'], '| hoegi:', S4['zoning']['zero3_share_by_zone_hoegi'])
    print(pd.crosstab(LAB[('hoegi','R1',0,8,0)][mh], zh.values[mh]).to_string())
else:
    S4['zoning'] = '보류: specialUse_dongDeamun.shp 없음'; print(S4['zoning'])

## 8. 그림 (f01~f08)

In [ ]:
import os; os.makedirs('/content/figs_f', exist_ok=True)
STY = {'R1': ('s', 'tab:blue'), 'R2': ('s', 'tab:gray'), 'A_fix': ('^', 'tab:red'), 'A_orig': ('v', 'darkred'), 'C': ('D', 'tab:purple')}
def plot_frontier(scale, fname, title):
    fig, axs = plt.subplots(1, len(KS), figsize=(4.2 * len(KS), 4), sharey=True)
    for ax, k in zip(axs, KS):
        pts = sorted((med(scale, 'R3', w, k, 'S')[0], med(scale, 'R3', w, k, 'sil')[0], w) for w in WS)
        ax.plot([p[0] for p in pts], [p[1] for p in pts], '-o', color='tab:green', ms=3, label='R3(w) 단순 절충')
        for c, (mk, col) in STY.items():
            p = 4 if c in ('A_fix', 'C') else 0
            if med(scale, c, p, k, 'S')[2]: ax.scatter(med(scale, c, p, k, 'S')[0], med(scale, c, p, k, 'sil')[0], marker=mk, color=col, s=45, label=c, zorder=3)
        for t in TOLS:
            if med(scale, 'B', t, k, 'S')[2]:
                ax.scatter(med(scale, 'B', t, k, 'S')[0], med(scale, 'B', t, k, 'sil')[0], marker='o', s=55 if t == 4 else 30,
                           color=plt.cm.autumn(TOLS.index(t) / 5), edgecolor='k' if t == 4 else 'none', label=f'B(tol={t})', zorder=4)
        ax.set_title(f'k={k}'); ax.set_xlabel('공간 응집 S'); ax.grid(alpha=.3)
    axs[0].set_ylabel('속성 실루엣'); axs[-1].legend(fontsize=7, loc='best'); fig.suptitle(title); fig.tight_layout()
    fig.savefig(f'/content/figs_f/{fname}', dpi=130); plt.close(fig)
plot_frontier('hoegi', 'f01_hoegi_frontier.png', '회기동: 속성 실루엣 vs 공간 응집 (시드 5개 중앙값)')
plot_frontier('gu', 'f02_gu_frontier.png', '동대문구: 속성 실루엣 vs 공간 응집 (시드 0, sil 5,000 표본)')
def plot_maps_profiles(scale, Xg, raw_df, Xn, fname, title):
    fig, axs = plt.subplots(2, 2, figsize=(13, 11))
    for col, (cand, p) in enumerate([('R1', 0), ('B', 4)]):
        L = LAB[(scale, cand, p, K_MAIN, 0)]
        axs[0, col].scatter(Xg[:, 0], Xg[:, 1], c=L, cmap='tab10', s=4 if len(L) > 5000 else 10); axs[0, col].set_aspect('equal')
        axs[0, col].set_title(f'{cand}{"(tol=4)" if cand == "B" else ""} k={K_MAIN} 시드 0'); axs[0, col].set_xticks([]); axs[0, col].set_yticks([])
        zdf = pd.DataFrame(Xn, columns=cols).groupby(L).mean()
        im = axs[1, col].imshow(zdf.values, cmap='RdBu_r', vmin=-2, vmax=2, aspect='auto')
        axs[1, col].set_xticks(range(len(cols))); axs[1, col].set_xticklabels(['용적률', '건폐율', '높이', '총층수', '건물나이'])
        axs[1, col].set_yticks(range(len(zdf))); axs[1, col].set_yticklabels([f'{i} (n={int((L == i).sum())})' for i in zdf.index])
        for (i, j), v in np.ndenumerate(zdf.values): axs[1, col].text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=axs[1, :], shrink=.6, label='z-score 평균'); fig.suptitle(title)
    fig.savefig(f'/content/figs_f/{fname}', dpi=130); plt.close(fig)
plot_maps_profiles('hoegi', X_geo, raw, X_ngf, 'f03_hoegi_maps_profiles.png', '회기동 k=8: 속성만(R1) vs 수정 GEO-SOM 노드 군집 B(4)')
plot_maps_profiles('gu', G_geo, G_raw, G_ngf, 'f04_gu_maps_profiles.png', '동대문구 k=8 (시드 0)')
# tol 다이얼, 컴포넌트 플레인, 덴드로그램, Gap 수정판
fig, axs = plt.subplots(1, 2, figsize=(11, 4))
for k in KS:
    axs[0].plot(range(len(TOLS)), EXPL['tol_dial'][k]['S'], '-o', label=f'k={k}'); axs[1].plot(range(len(TOLS)), EXPL['tol_dial'][k]['sil'], '-o', label=f'k={k}')
for a, t in zip(axs, ['공간 응집 S', '속성 실루엣']): a.set_xticks(range(len(TOLS))); a.set_xticklabels(['0', '2', '4', '8', '∞']); a.set_xlabel('tol'); a.set_title(t); a.grid(alpha=.3)
axs[1].legend(fontsize=8); fig.suptitle('회기동 B(tol): tol 다이얼 (시드 5개 중앙값)'); fig.tight_layout(); fig.savefig('/content/figs_f/f05_tol_dial.png', dpi=130); plt.close(fig)
m0 = SOMS[('hoegi', 4, 0, (20, 20))][0]
fig, axs = plt.subplots(1, 5, figsize=(18, 3.6))
for i, (a, nm) in enumerate(zip(axs, ['용적률', '건폐율', '높이', '총층수', '건물나이'])):
    im = a.imshow(m0.map_ngf[:, i].reshape(20, 20), origin='lower', cmap='RdBu_r', vmin=-2, vmax=2); a.set_title(nm); a.set_xticks([]); a.set_yticks([])
fig.colorbar(im, ax=axs, shrink=.8); fig.suptitle('회기동 수정 GEO-SOM(tol=4, 시드 0) 컴포넌트 플레인 (행=남→북, 열=서→동 격자)')
fig.savefig('/content/figs_f/f06_component_planes.png', dpi=130); plt.close(fig)
from scipy.cluster.hierarchy import linkage, dendrogram
fig, ax = plt.subplots(figsize=(12, 4)); dendrogram(linkage(m0.map_ngf, 'ward'), truncate_mode='lastp', p=30, ax=ax)
ax.set_title('회기동 B/C 노드 가중치 Ward 덴드로그램 (tol=4, 시드 0, 마지막 30개 병합)'); fig.savefig('/content/figs_f/f07_dendrogram.png', dpi=130); plt.close(fig)
fig, axs = plt.subplots(2, 2, figsize=(12, 7))
for r_, name in enumerate(GAP):
    for c_, mc in enumerate((15, 30)):
        for d in GAP[name][mc]: axs[r_, c_].errorbar(range(1, mc), d['gap'], yerr=d['sk'], alpha=.6, capsize=2)
        axs[r_, c_].set_title(f'{name} maxClusters={mc}: k={[d["k"] for d in GAP[name][mc]]}'); axs[r_, c_].grid(alpha=.3)
fig.suptitle('Gap 수정판 (PCA 범위 기준, nrefs=20, 1-SE)'); fig.tight_layout(); fig.savefig('/content/figs_f/f08_gap_fixed.png', dpi=130); plt.close(fig)
print(sorted(os.listdir('/content/figs_f')))   # 실행본은 plt.show() 대신 plt.close(fig) (출력 크기 줄임)

## 9. 결과 저장

In [ ]:
# 결과 요약 JSON (/content/results_analysis_f.json)
if 'zoning' not in S4: S4['zoning'] = '용도지역 파일 없음 (건너뜀)'
RESULTS_F = dict(
    meta=dict(N_hoegi=int(N), N_gu=int(GN), KS=KS, K_MAIN=K_MAIN, SEEDS=SEEDS, WS=WS, TOLS=TOLS, check_vectorized=CHECK_V),
    V1=V1, V2=V2, V3=V3, V4=V4, V5=S3['v5'], V6=V6, V7=V7,
    EXPL=EXPL, S1=S1, S3=S3, S4=S4, SIL_ARGMAX={n: v['sil_argmax'] for n, v in SIL_ELBOW.items()}, gap=GAP,
    medians=df_rows().groupby(['scale', 'cand', 'param', 'k'])[['sil', 'r2', 'db', 'S', 'lift', 'k_eff']].median().round(4).reset_index().to_dict(orient='records'))
json.dump(RESULTS_F, open('/content/results_analysis_f.json', 'w'), ensure_ascii=False, default=float, indent=1)
print('저장: /content/results_analysis_f.json')

## 10. 그림으로 점검 (5단계)

주장마다 대응하는 그림을 그리다가 드러난 두 가지를 확인합니다. 아래 셀은 건물이 하나도 붙지 않은 노드와, 한 번도 갱신되지 않은(초기 난수 그대로인) 노드 수를 셉니다.

In [ ]:
def node_check(scale, tol=4, seed=0):
    m, bmu = SOMS[(scale, tol, seed, (20, 20))]
    hits = np.bincount(bmu, minlength=400)
    used = np.where(hits > 0)[0]
    dmin = np.min(np.linalg.norm(m.grid_pos[:, None] - m.grid_pos[used][None], axis=2), axis=1)
    untouched = dmin > 2
    inrange = ((m.map_ngf >= 0) & (m.map_ngf <= 1)).all(1)
    return dict(n_used=int(len(used)), n_untouched=int(untouched.sum()), untouched_all_in_0_1=bool(inrange[untouched].all()) if untouched.any() else None,
                n_unused=int((hits == 0).sum()))
NODE_CHECK = {f'{sc}_tol{t}': node_check(sc, t) for sc, t in [('hoegi', 4), ('hoegi', 0), ('hoegi', 30), ('gu', 4)]}
m, bmu = SOMS[('hoegi', 4, 0, (20, 20))]; hits = np.bincount(bmu, minlength=400)
nl = node_labels_km(m.map_ngf, 8, 0)
NODE_CHECK['hoegi_B4_k8_nodes_per_cluster'] = {int(c): dict(nodes=int((nl == c).sum()), used_nodes=int(((nl == c) & (hits > 0)).sum()), buildings=int(hits[nl == c].sum())) for c in range(8)}
print(json.dumps(NODE_CHECK, ensure_ascii=False))

### 10-1. 추가 그림 (f09~f16)

후보별 지도, 0 값 건물의 위치와 분포, 군집별 0 값 비율, 외부 변수, 안정성, 군집별 지도, 노드별 건물 수.

In [ ]:
KO = {'A20': '용적률', 'A21': '건폐율', 'A30': '높이', 'total_floor': '층수', 'years_old': '나이'}
def ko(name):
    for a, b in KO.items(): name = name.replace(a, b)
    return name
def mapax(ax, Xg, L, title):
    ax.scatter(Xg[:, 0], Xg[:, 1], c=L, cmap='tab10', vmin=-0.5, vmax=9.5, s=2 if len(L) > 5000 else 8, linewidths=0)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=10)
def row_of(sc, c, p, k, s): return [x for x in ROWS if x['scale'] == sc and x['cand'] == c and x['param'] == p and x['k'] == k and x['seed'] == s][0]

# f09 판정 2를 눈으로: 후보별 지도
fig, axs = plt.subplots(2, 3, figsize=(15, 11))
spec = [('R1', 0, 'R1 속성만'), ('R2', 0, 'R2 좌표만'), ('A_orig', 0, 'A_orig 원 설계 [x,y,x′,y′] (원본 17군집 위)'),
        ('A_fix', 4, 'A_fix 원 설계 (수정본 위)'), ('B', 4, 'B(tol=4) 수정 GEO-SOM 노드 군집'), ('B', 0, 'B(tol=0) 지리 칸 그대로')]
for ax, (c, p, t) in zip(axs.ravel(), spec):
    L = LAB[('hoegi', c, p, 8, 0)]; r = row_of('hoegi', c, p, 8, 0)
    mapax(ax, X_geo, L, f"{t}\nsil={r['sil']:.2f}, S={r['S']:.2f}, ARI(vs R2)={adjusted_rand_score(L, LAB[('hoegi', 'R2', 0, 8, 0)]):.2f}")
fig.suptitle('회기동 k=8 시드 0: 후보별 군집 지도 (색은 각 그림 안에서만 의미)')
fig.tight_layout(); fig.savefig('/content/figs_f/f09_hoegi_maps_compare.png', dpi=120); plt.close(fig)

# f10 0 값
fig = plt.figure(figsize=(16, 10))
for i, (Xg, z, nm) in enumerate([(X_geo, zero3, '회기동'), (G_geo, zero3g, '동대문구')]):
    ax = fig.add_subplot(2, 3, 1 + i); s = 2 if len(z) > 5000 else 8
    ax.scatter(Xg[~z, 0], Xg[~z, 1], c='lightgray', s=s, linewidths=0, label=f'값 있음 {int((~z).sum())}')
    ax.scatter(Xg[z, 0], Xg[z, 1], c='crimson', s=s, linewidths=0, label=f'용적률·건폐율·높이 모두 0: {int(z.sum())} ({z.mean():.0%})')
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f'{nm}: 0 묶음 위치'); ax.legend(fontsize=8, loc='lower left', markerscale=3)
ax = fig.add_subplot(2, 3, 3); jit = np.random.default_rng(0).uniform(-0.25, 0.25, len(G_raw))
ax.scatter(G_raw['total_floor'] + jit, G_raw['A30'], s=2, alpha=.3, c=np.where(zero3g, 'crimson', 'tab:blue'), linewidths=0)
ax.set_xlabel('총층수 (지상+지하, 지터)'); ax.set_ylabel('높이 A30 (m)'); ax.set_title('동대문구: 층수는 있는데 높이 0 (빨강 = 0 묶음)'); ax.set_xlim(0, 25); ax.set_ylim(-2, 90)
for i, (c, nm) in enumerate([('A20', '용적률 (%)'), ('A21', '건폐율 (%)'), ('A30', '높이 (m)')]):
    ax = fig.add_subplot(2, 3, 4 + i); v = G_raw[c]; hi = v.quantile(.99)
    ax.hist(v[v > 0].clip(upper=hi), bins=60, color='tab:blue', alpha=.8, label='0 초과')
    ax.bar([0], [(v == 0).sum()], width=hi / 60, color='crimson', label=f'0: {int((v == 0).sum())}')
    ax.set_yscale('log'); ax.set_title(f'동대문구 {nm} 분포 (로그 축, 상위 1% 절단)'); ax.legend(fontsize=8)
fig.suptitle('용적률·건폐율·높이 0 값 (결측 대용 의심)'); fig.tight_layout()
fig.savefig('/content/figs_f/f10_zero_values.png', dpi=120); plt.close(fig)

# f11 군집별 0 묶음 비율
fig, axs = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, (sc, c, p, z, nm) in zip(axs, [('hoegi', 'R1', 0, zero3, '회기동 R1'), ('hoegi', 'B', 4, zero3, '회기동 B(4)'), ('gu', 'R1', 0, zero3g, '동대문구 R1'), ('gu', 'B', 4, zero3g, '동대문구 B(4)')]):
    L = LAB[(sc, c, p, 8, 0)]; u = np.unique(L); share = [z[L == i].mean() for i in u]; n = [(L == i).sum() for i in u]
    ax.bar(range(len(u)), share, color=['crimson' if s_ >= .8 else 'tab:blue' for s_ in share])
    for j, (s_, n_) in enumerate(zip(share, n)): ax.text(j, s_ + .02, f'n={n_}', ha='center', fontsize=7)
    ax.set_xticks(range(len(u))); ax.set_xticklabels(u); ax.set_title(nm); ax.set_xlabel('군집 (k=8, 시드 0)')
axs[0].set_ylabel('군집 안 0 묶음 비율'); axs[0].set_ylim(0, 1.12)
fig.suptitle('군집별 0 묶음 비율 (빨강: 80% 이상)'); fig.tight_layout(); fig.savefig('/content/figs_f/f11_zero_share_by_cluster.png', dpi=120); plt.close(fig)

# f12 외부 변수 (용도지역 파일이 있을 때만)
if HAS_ZONING:
    ZC = {'1종일반': '#ffe680', '2종일반': '#f4b400', '3종일반': '#e67c00', '준주거': '#c2185b', '상업': '#d32f2f', '자연녹지': '#2e7d32'}
    fig = plt.figure(figsize=(17, 10))
    for i, (Xg, zz, nm) in enumerate([(X_geo, zh.values, '회기동'), (G_geo, zg.values, '동대문구')]):
        ax = fig.add_subplot(2, 2, 1 + i); s = 2 if len(zz) > 5000 else 8
        for k_, col in ZC.items():
            m_ = zz == k_
            if m_.any(): ax.scatter(Xg[m_, 0], Xg[m_, 1], c=col, s=s, linewidths=0, label=f'{k_} {int(m_.sum())}')
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f'{nm}: 건물 centroid의 용도지역 (6분류)'); ax.legend(fontsize=8, markerscale=3, loc='lower left')
    ax = fig.add_subplot(2, 1, 2)
    items = [('주용도 회기동', S4['V_use'], 'hoegi'), ('주용도 동대문구', S4['V_use'], 'gu'), ('법정동 동대문구', {c: {'gu': v} for c, v in S4['V_dong_gu'].items()}, 'gu'),
             ('용도지역 회기동', S4['zoning']['V'], 'hoegi'), ('용도지역 동대문구', S4['zoning']['V'], 'gu')]
    x = np.arange(len(items)); wdt = .26
    for j, (c, col) in enumerate([('R1', 'tab:blue'), ('R2', 'tab:gray'), ('B', 'tab:orange')]):
        vals = [d[c][sc] for _, d, sc in items]
        ax.bar(x + (j - 1) * wdt, vals, wdt, color=col, label={'R1': 'R1 속성만', 'R2': 'R2 좌표만', 'B': 'B(tol=4)'}[c])
        for xi, v in zip(x + (j - 1) * wdt, vals): ax.text(xi, v + .01, f'{v:.2f}', ha='center', fontsize=8)
    ax.axhline(0.3, ls='--', c='k', lw=.8); ax.text(len(items) - .5, .31, '읽는 규칙 0.3 (용도지역 재발견)', ha='right', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels([t for t, _, _ in items]); ax.set_ylabel("Cramér's V (k=8)"); ax.legend()
    fig.suptitle('외부 변수와 군집의 관계'); fig.tight_layout(); fig.savefig('/content/figs_f/f12_external.png', dpi=120); plt.close(fig)

# f13 안정성·SOM 품질
fig, axs = plt.subplots(1, 3, figsize=(17, 4.2))
for c, p, lab in [('R1', 0, 'R1'), ('R2', 0, 'R2'), ('B', 4, 'B(4)'), ('C', 4, 'C(4)'), ('A_fix', 4, 'A_fix'), ('B', 0, 'B(0)'), ('B', 30, 'B(∞)')]:
    axs[0].plot(KS, [ari_within('hoegi', c, p, k) for k in KS], '-o', label=lab)
axs[0].axhline(.75, ls='--', c='k', lw=.7); axs[0].axhline(.5, ls=':', c='k', lw=.7)
axs[0].set_xlabel('k'); axs[0].set_ylabel('시드 쌍별 ARI 중앙값'); axs[0].set_title('회기동 안정성 (선: 0.75 / 0.5)'); axs[0].legend(fontsize=7)
tl = ['0', '2', '4', '8', '∞']
axs[1].plot(tl, [S1['qe_te'][t]['qe'] for t in TOLS], '-o'); axs[1].set_title('양자화 오차 QE (tol별, 20×20)'); axs[1].set_xlabel('tol')
axs[2].plot(tl, [S1['qe_te'][t]['te'] for t in TOLS], '-o', c='tab:red'); axs[2].set_title('위상 오차 TE (8-이웃)'); axs[2].set_xlabel('tol')
fig.suptitle(f"회기동 SOM 품질·안정성 (tol=4 epochs 50→100 QE 변화 {S1['epochs']['rel_change']:.0%})"); fig.tight_layout()
fig.savefig('/content/figs_f/f13_stability_quality.png', dpi=120); plt.close(fig)

# f14·f15 R1 k=8 군집마다 따로 그린 지도
def small_multiples(sc, Xg, Xn, rawdf, z, fname, title):
    L = LAB[(sc, 'R1', 0, 8, 0)]; prof = profiles(rawdf, Xn, L)
    fig, axs = plt.subplots(2, 4, figsize=(18, 10))
    for ax, i in zip(axs.ravel(), np.unique(L)):
        m_ = L == i; s = 1.5 if len(L) > 5000 else 6
        ax.scatter(Xg[~m_, 0], Xg[~m_, 1], c='#dddddd', s=s, linewidths=0)
        ax.scatter(Xg[m_, 0], Xg[m_, 1], c='crimson', s=s * 2, linewidths=0)
        pr = prof.loc[str(i)]
        ax.set_title(f"{i}: n={int(pr['n'])}, 0묶음 {z[m_].mean():.0%}\n{ko(pr['name'])}\n용적률 {pr['A20']:.0f} 건폐율 {pr['A21']:.0f} 높이 {pr['A30']:.1f}m {pr['total_floor']:.1f}층 {pr['years_old']:.0f}년", fontsize=8)
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(title); fig.tight_layout(); fig.savefig(f'/content/figs_f/{fname}', dpi=120); plt.close(fig)
small_multiples('hoegi', X_geo, X_ngf, raw, zero3, 'f14_hoegi_R1_small_multiples.png', '회기동 R1(속성만) k=8 시드 0: 군집별 위치 (빨강), 이름은 |z|≥0.5 변수, 값은 원값 평균')
small_multiples('gu', G_geo, G_ngf, G_raw, zero3g, 'f15_gu_R1_small_multiples.png', '동대문구 R1(속성만) k=8 시드 0: 군집별 위치 (빨강)')

# f16 노드 적중 수와 학습 안 된 노드
fig, axs = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, (sc, nm) in zip(axs[:2], [('hoegi', '회기동'), ('gu', '동대문구')]):
    m, bmu = SOMS[(sc, 4, 0, (20, 20))]; h2 = np.bincount(bmu, minlength=400).reshape(20, 20).astype(float)
    im = ax.imshow(np.where(h2 > 0, h2, np.nan), origin='lower', cmap='viridis'); ax.set_title(f'{nm} tol=4 시드 0: 노드별 건물 수 (흰 칸 = 0)', fontsize=10); fig.colorbar(im, ax=ax, shrink=.8)
m, bmu = SOMS[('hoegi', 4, 0, (20, 20))]; nl2 = node_labels_km(m.map_ngf, 8, 0).reshape(20, 20); h2 = np.bincount(bmu, minlength=400).reshape(20, 20)
axs[2].imshow(nl2, origin='lower', cmap='tab10', vmin=-.5, vmax=9.5)
yy, xx = np.where(h2 == 0); axs[2].scatter(xx, yy, marker='x', c='k', s=8, linewidths=.6)
axs[2].set_title('회기동 B(4) k=8 노드 군집 (x = 건물 없는 노드)', fontsize=10)
for a in axs: a.set_xticks([]); a.set_yticks([])
nc = NODE_CHECK['hoegi_tol4']
fig.suptitle(f"학습 안 된 노드: 회기동 400개 중 {nc['n_unused']}개가 건물 0, 그중 {nc['n_untouched']}개는 초기 난수 그대로 (격자: 아래=남, 왼쪽=서)")
fig.tight_layout(); fig.savefig('/content/figs_f/f16_node_hits.png', dpi=120); plt.close(fig)

print(sorted(os.listdir('/content/figs_f')))

## 11. 바로잡고 다시 판정 (6단계)

- **D1:** 용적률·건폐율·높이 중 하나라도 0인 건물을 제외(값이 없음을 0으로 채운 것으로 보임). 남은 건물로 다시 표준화하고 SOM 범위·이웃도 다시 만듭니다.
- **B_hit:** 건물이 1개 이상 붙은 노드만, 건물 수를 가중치로 K-Means.

먼저 0 값 패턴, 대장 구분(A12)·주용도별 0 비율, 0이 아닌 건물의 높이–층수 상관을 봅니다.

In [ ]:
def node_labels_hit(W, hits, k, seed):
    used = np.where(hits > 0)[0]
    lab = KMeans(n_clusters=k, n_init=10, random_state=seed).fit_predict(W[used], sample_weight=hits[used])
    nl = np.full(len(W), -1); nl[used] = lab; return nl
def node_labels_ward_hit(W, hits, k):
    used = np.where(hits > 0)[0]
    lab = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(W[used])
    nl = np.full(len(W), -1); nl[used] = lab; return nl
def run_hit(scale, Xg, Xn, nn, seeds, tols, epochs=50, sample=None, do_all=True, do_C=False):
    for tol in tols:
        for s in seeds:
            t0 = time.time(); key = (scale, tol, s, (20, 20))
            if key in SOMS: m, bmu = SOMS[key]
            else:
                m, bmu = train_som(Xg, Xn, tol, s, epochs=epochs); SOMS[key] = (m, bmu)
            hits = np.bincount(bmu, minlength=len(m.map_ngf)); W = m.map_ngf
            for k in KS:
                record(scale, 'B_hit', tol, k, s, node_labels_hit(W, hits, k, s)[bmu], Xn, nn, sample)
                if do_all and tol == 4: record(scale, 'B_all', tol, k, s, node_labels_km(W, k, s)[bmu], Xn, nn, sample)
                if do_C and tol == 4: record(scale, 'C_hit', tol, k, s, node_labels_ward_hit(W, hits, k)[bmu], Xn, nn, sample)
            print(f'{scale} tol={tol} seed={s}: {time.time()-t0:.1f}s, 적중 노드={int((hits > 0).sum())}')

mask_h = (raw[['A20', 'A21', 'A30']] != 0).all(1).values
mask_g = (G_raw[['A20', 'A21', 'A30']] != 0).all(1).values
reg_g = dongDea_gdf.loc[GP.index, 'A12'].astype(str).values
anyz_g = ~mask_g
OBS = {}
def zero_pattern(df):
    z = (df[['A20', 'A21', 'A30']] == 0)
    return z.apply(lambda r: ''.join('0' if v else '.' for v in r), axis=1).value_counts().to_dict()   # 순서: 용적률·건폐율·높이, '0'=0값
OBS['O1'] = dict(hoegi=zero_pattern(raw), gu=zero_pattern(G_raw), N_D1_hoegi=int(mask_h.sum()), N_D1_gu=int(mask_g.sum()))
OBS['O2'] = dict(by_register={r: dict(n=int((reg_g == r).sum()), anyzero=float(anyz_g[reg_g == r].mean())) for r in np.unique(reg_g)},
                 by_use_gu={u: dict(n=int((use_g == u).sum()), anyzero=float(anyz_g[use_g == u].mean())) for u in pd.Series(use_g).value_counts().index[:10]})
OBS['O3'] = dict(corr_height_floor_D1_hoegi=float(np.corrcoef(raw['A30'][mask_h], raw['total_floor'][mask_h])[0, 1]),
                 corr_height_floor_D1_gu=float(np.corrcoef(G_raw['A30'][mask_g], G_raw['total_floor'][mask_g])[0, 1]),
                 corr_height_floor_D0_gu=float(np.corrcoef(G_raw['A30'], G_raw['total_floor'])[0, 1]))
print(json.dumps(OBS, ensure_ascii=False, default=float))

### 11-1. 빈 노드 효과 (0 값 포함 데이터)

기준: 5개 k 중 3개 이상에서 실루엣(B_hit(4)) − 실루엣(B_all(4)) > 0.02이면 "빈 노드 효과 있음", ±0.02 이내가 3개 이상이면 "효과 없음", 그 외 "보류". B_hit(4)를 절충 곡선과도 비교합니다(판정 1과 같은 규칙).

In [ ]:
run_hit('hoegi', X_geo, X_ngf, nn_h, SEEDS, [0, 4, 30], do_all=False, do_C=True)
F2 = {}
diffs = {k: med('hoegi', 'B_hit', 4, k, 'sil')[0] - med('hoegi', 'B', 4, k, 'sil')[0] for k in KS}
up = sum(d > 0.02 for d in diffs.values()); flat = sum(abs(d) <= 0.02 for d in diffs.values())
keff_ok = all(r['k_eff'] == r['k'] for r in ROWS if r['scale'] == 'hoegi' and r['cand'] == 'B_hit')
F2['f2_1'] = dict(verdict='빈 노드 효과 있음' if up >= 3 else ('빈 노드 효과 없음' if flat >= 3 else '보류'), diffs=diffs, keff_all_equal_k=keff_ok,
                  vs_curve_D0=verdict1('hoegi', 'B_hit', 4))
print('f2-1:', F2['f2_1']['verdict'], {k: round(v, 3) for k, v in diffs.items()}, '| k_eff=k:', keff_ok)
print('   D0 B_hit(4) vs 곡선:', F2['f2_1']['vs_curve_D0']['verdict'], [(d['cls'], None if d['delta'] is None else round(d['delta'], 3)) for d in F2['f2_1']['vs_curve_D0']['per_k']], round(F2['f2_1']['vs_curve_D0']['stab_ari'], 3))

### 11-2. 결측을 뺀 데이터(D1)에서 재판정 (핵심)

- B_hit(4) vs D1 절충 곡선: 판정 1과 같은 규칙.
- 결측이 공간 응집을 부풀렸나: 5개 k 중 3개 이상에서 S(R1, D1) ≤ S(R1, 0 값 포함) − 0.05이면 "부풀림 확인", |차이| < 0.05가 3개 이상이면 "부풀림 없음".
- epochs 100으로 다시 학습해 판정 이름이 같으면 "학습 길이에 강건".

In [ ]:
Xg1 = X_geo[mask_h]; raw1 = raw[mask_h]
Xn1 = StandardScaler().fit_transform(MinMaxScaler().fit_transform(raw1.values)); nn1 = make_nn(Xg1)
print('회기동 D1 N =', len(Xg1))
t0 = time.time(); run_baselines('hoegiD1', Xg1, Xn1, nn1, SEEDS); print(f'기준선 {time.time()-t0:.0f}s')
run_hit('hoegiD1', Xg1, Xn1, nn1, SEEDS, [0, 4, 30], do_all=True, do_C=True)
F2['f2_2'] = verdict1('hoegiD1', 'B_hit', 4)
print('f2-2:', F2['f2_2']['verdict'], [(d['cls'], None if d['delta'] is None else round(d['delta'], 3)) for d in F2['f2_2']['per_k']], '| 안정성', round(F2['f2_2']['stab_ari'], 3), F2['f2_2']['stab'])
sd = {k: med('hoegiD1', 'R1', 0, k, 'S')[0] - med('hoegi', 'R1', 0, k, 'S')[0] for k in KS}
F2['f2_3'] = dict(verdict='부풀림 확인' if sum(d <= -0.05 for d in sd.values()) >= 3 else ('부풀림 없음' if sum(abs(d) < 0.05 for d in sd.values()) >= 3 else '보류'),
                  S_D0={k: med('hoegi', 'R1', 0, k, 'S')[0] for k in KS}, S_D1={k: med('hoegiD1', 'R1', 0, k, 'S')[0] for k in KS}, diff=sd)
print('f2-3:', F2['f2_3']['verdict'], {k: round(v, 3) for k, v in sd.items()})
run_hit('hoegiD1e100', Xg1, Xn1, nn1, SEEDS, [4], epochs=100, do_all=False)
F2['f2_4'] = verdict1('hoegiD1e100', 'B_hit', 4, curve_scale='hoegiD1'); F2['f2_4']['same_as_f2_2'] = F2['f2_4']['verdict'] == F2['f2_2']['verdict']
print('f2-4:', F2['f2_4']['verdict'], '->', '학습 길이에 강건' if F2['f2_4']['same_as_f2_2'] else '학습 길이에 따라 다름')
F2['expl_D1'] = {f'{c}({p})': [delta('hoegiD1', c, p, k)['cls'] for k in KS] for c, p in [('B_hit', 0), ('B_hit', 30), ('B_all', 4), ('C_hit', 4)]}
F2['stab_D1'] = {f'{c}({p})': float(np.median([ari_within('hoegiD1', c, p, k) for k in KS])) for c, p in [('R1', 0), ('R2', 0), ('B_hit', 4), ('B_all', 4), ('C_hit', 4)]}
print(json.dumps({k: F2[k] for k in ('expl_D1', 'stab_D1')}, ensure_ascii=False))
print(df_rows().query("scale=='hoegiD1'").groupby(['cand', 'param', 'k'])[['sil', 'S']].median().round(3).unstack('k').to_string())

### 11-3. 동대문구 D1 (시드 0)

규칙은 6장과 같습니다("비교 불가" 일치는 세지 않음).

In [ ]:
Gg1 = G_geo[mask_g]; Graw1 = G_raw[mask_g]
Gn1 = StandardScaler().fit_transform(MinMaxScaler().fit_transform(Graw1.values)); nng1 = make_nn(Gg1)
print('동대문구 D1 N =', len(Gg1))
t0 = time.time(); run_baselines('guD1', Gg1, Gn1, nng1, [0], sample=5000); print(f'기준선 {time.time()-t0:.0f}s')
run_hit('guD1', Gg1, Gn1, nng1, [0], [0, 4, 30], sample=5000, do_all=True)
rows5 = []
for k in KS:
    h = delta('hoegiD1', 'B_hit', 4, k)['cls']; g = delta('guD1', 'B_hit', 4, k, eps_fixed=0.02)
    rows5.append(dict(k=k, hoegi=h, gu=g['cls'], gu_delta=g['delta'], gu_S=g['S'], gu_sil=g['sil']))
F2['f2_5'] = dict(per_k=rows5, same=sum(r['hoegi'] == r['gu'] for r in rows5))
F2['f2_5']['verdict'] = '구 규모에서도 같은 방향' if F2['f2_5']['same'] >= 3 else '규모에 따라 다름'
print('f2-5:', F2['f2_5'])
print(df_rows().query("scale=='guD1'").groupby(['cand', 'param', 'k'])[['sil', 'S']].median().round(3).unstack('k').to_string())

### 11-4. D1 프로파일과 외부 변수

In [ ]:
INS = {}
for cand, p in [('R1', 0), ('B_hit', 4)]:
    INS[f'{cand}{p}'] = dict(prof_seed0=profiles(raw1, Xn1, LAB[('hoegiD1', cand, p, 8, 0)]).to_dict(orient='index'),
                             recur=recur('hoegiD1', cand, p, 8, Xn1).to_dict(orient='records'),
                             gu_names=sorted(set(profiles(Graw1, Gn1, LAB[('guD1', cand, p, 8, 0)])['name'])),
                             gu_prof=profiles(Graw1, Gn1, LAB[('guD1', cand, p, 8, 0)]).to_dict(orient='index'))
EXT = {}
for c, p in [('R1', 0), ('R2', 0), ('B_hit', 4)]:
    EXT[c] = dict(zoning_hoegi=float(np.median([cramers_v(LAB[('hoegiD1', c, p, 8, s)], zh.values[mask_h]) for s in SEEDS])) if HAS_ZONING else None,
                  zoning_gu=cramers_v(LAB[('guD1', c, p, 8, 0)], zg.values[mask_g]) if HAS_ZONING else None,
                  use_hoegi=float(np.median([cramers_v(LAB[('hoegiD1', c, p, 8, s)], use_h[mask_h]) for s in SEEDS])),
                  use_gu=cramers_v(LAB[('guD1', c, p, 8, 0)], use_g[mask_g]),
                  dong_gu=cramers_v(LAB[('guD1', c, p, 8, 0)], dong_g[mask_g]))
F2['ins'] = INS; F2['ext_D1'] = EXT
print(json.dumps(EXT, ensure_ascii=False))
pd.set_option('display.width', 250)
for k_ in INS:
    print(k_, 'recur:', INS[k_]['recur']); print('   gu names:', INS[k_]['gu_names'])
    print(pd.DataFrame(INS[k_]['prof_seed0']).T.to_string())

### 11-5. 그림 (g01~g12)

In [ ]:
import os; os.makedirs('/content/figs_f2', exist_ok=True)
def frontier2(curve_scale, pts, fname, title):
    fig, axs = plt.subplots(1, len(KS), figsize=(4.2 * len(KS), 4.2), sharey=True)
    for ax, k in zip(axs, KS):
        cv = sorted((med(curve_scale, 'R3', w, k, 'S')[0], med(curve_scale, 'R3', w, k, 'sil')[0]) for w in WS)
        ax.plot([p[0] for p in cv], [p[1] for p in cv], '-o', color='tab:green', ms=3, label='R3(w) 단순 절충')
        for sc, c, p, lab, mk, col, sz in pts:
            if med(sc, c, p, k, 'S')[2]:
                ax.scatter(med(sc, c, p, k, 'S')[0], med(sc, c, p, k, 'sil')[0], marker=mk, color=col, s=sz, edgecolor='k' if sz > 50 else 'none', label=lab, zorder=4)
        ax.set_title(f'k={k}'); ax.set_xlabel('공간 응집 S'); ax.grid(alpha=.3)
    axs[0].set_ylabel('속성 실루엣'); axs[-1].legend(fontsize=7, loc='best'); fig.suptitle(title); fig.tight_layout()
    fig.savefig(f'/content/figs_f2/{fname}', dpi=120); plt.close(fig)
P_R = [('R1', 0, 'R1 속성만', 's', 'tab:blue', 45), ('R2', 0, 'R2 좌표만', 's', 'tab:gray', 45)]
frontier2('hoegi', [('hoegi',) + t for t in P_R] + [('hoegi', 'B', 4, 'B_all(4) = (f)의 B', 'o', 'tab:orange', 45), ('hoegi', 'B_hit', 4, 'B_hit(4) 판정 대상', 'o', 'crimson', 70),
          ('hoegi', 'B_hit', 0, 'B_hit(0)', 'v', 'salmon', 30), ('hoegi', 'B_hit', 30, 'B_hit(∞)', '^', 'gold', 30), ('hoegi', 'C_hit', 4, 'C_hit(4)', 'D', 'tab:purple', 30)],
          'g01_hoegi_D0_frontier_hit.png', '회기동 D0(0 값 포함): 빈 노드를 뺀 B_hit vs 단순 절충 (시드 5개 중앙값)')
frontier2('hoegiD1', [('hoegiD1',) + t for t in P_R] + [('hoegiD1', 'B_all', 4, 'B_all(4)', 'o', 'tab:orange', 45), ('hoegiD1', 'B_hit', 4, 'B_hit(4) 판정 대상', 'o', 'crimson', 70),
          ('hoegiD1e100', 'B_hit', 4, 'B_hit(4) epochs 100', 'o', 'darkred', 35), ('hoegiD1', 'B_hit', 0, 'B_hit(0)', 'v', 'salmon', 30), ('hoegiD1', 'B_hit', 30, 'B_hit(∞)', '^', 'gold', 30),
          ('hoegiD1', 'C_hit', 4, 'C_hit(4)', 'D', 'tab:purple', 30)],
          'g02_hoegi_D1_frontier.png', f'회기동 D1(0 값 건물 제외, N={len(Xg1)}): 속성 실루엣 vs 공간 응집 (시드 5개 중앙값)')
frontier2('guD1', [('guD1',) + t for t in P_R] + [('guD1', 'B_all', 4, 'B_all(4)', 'o', 'tab:orange', 45), ('guD1', 'B_hit', 4, 'B_hit(4)', 'o', 'crimson', 70),
          ('guD1', 'B_hit', 0, 'B_hit(0)', 'v', 'salmon', 30), ('guD1', 'B_hit', 30, 'B_hit(∞)', '^', 'gold', 30)],
          'g03_gu_D1_frontier.png', f'동대문구 D1(N={len(Gg1)}, 시드 0, sil 5,000 표본)')

def mapax2(ax, Xg, L, title):
    ax.scatter(Xg[:, 0], Xg[:, 1], c=L, cmap='tab10', vmin=-0.5, vmax=9.5, s=2 if len(L) > 5000 else 10, linewidths=0)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(title, fontsize=10)
fig, axs = plt.subplots(2, 3, figsize=(15, 11))
for ax, (c, p, t) in zip(axs.ravel(), [('R1', 0, 'R1 속성만'), ('R2', 0, 'R2 좌표만'), ('B_hit', 4, 'B_hit(4) 판정 대상'), ('B_all', 4, 'B_all(4)'), ('B_hit', 0, 'B_hit(0) 지리 칸'), ('C_hit', 4, 'C_hit(4) Ward')]):
    L = LAB[('hoegiD1', c, p, 8, 0)]; r = row_of('hoegiD1', c, p, 8, 0)
    mapax2(ax, Xg1, L, f"{t}\nsil={r['sil']:.2f}, S={r['S']:.2f}, ARI(vs R1)={adjusted_rand_score(L, LAB[('hoegiD1', 'R1', 0, 8, 0)]):.2f}")
fig.suptitle(f'회기동 D1 k=8 시드 0 (0 값 건물 {int((~mask_h).sum())}개 제외, 색은 각 그림 안에서만 의미)')
fig.tight_layout(); fig.savefig('/content/figs_f2/g04_hoegi_D1_maps_compare.png', dpi=120); plt.close(fig)

def sm2(sc, Xg, Xn, rawdf, Xall, fname, title):
    L = LAB[(sc, 'R1', 0, 8, 0)]; prof = profiles(rawdf, Xn, L)
    fig, axs = plt.subplots(2, 4, figsize=(18, 10))
    for ax, i in zip(axs.ravel(), np.unique(L)):
        m_ = L == i; s = 1.5 if len(L) > 5000 else 6
        ax.scatter(Xall[:, 0], Xall[:, 1], c='#f0f0f0', s=s, linewidths=0)
        ax.scatter(Xg[~m_, 0], Xg[~m_, 1], c='#c8c8c8', s=s, linewidths=0)
        ax.scatter(Xg[m_, 0], Xg[m_, 1], c='crimson', s=s * 2, linewidths=0)
        pr = prof.loc[str(i)]
        ax.set_title(f"{i}: n={int(pr['n'])}\n{ko(pr['name'])}\n용적률 {pr['A20']:.0f} 건폐율 {pr['A21']:.0f} 높이 {pr['A30']:.1f}m {pr['total_floor']:.1f}층 {pr['years_old']:.0f}년", fontsize=8)
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(title); fig.tight_layout(); fig.savefig(f'/content/figs_f2/{fname}', dpi=120); plt.close(fig)
sm2('hoegiD1', Xg1, Xn1, raw1, X_geo, 'g05_hoegi_D1_small_multiples.png', '회기동 D1 R1 k=8 시드 0: 군집별 위치 (빨강, 연회색 = 제외된 0 값 건물)')
sm2('guD1', Gg1, Gn1, Graw1, G_geo, 'g06_gu_D1_small_multiples.png', '동대문구 D1 R1 k=8 시드 0: 군집별 위치 (빨강, 연회색 = 제외된 0 값 건물)')

fig, axs = plt.subplots(1, 4, figsize=(26, 6)); fig.subplots_adjust(wspace=0.55, left=0.06, right=0.9)
for ax, (sc, Xn_, c, p, t) in zip(axs, [('hoegiD1', Xn1, 'R1', 0, '회기동 R1'), ('hoegiD1', Xn1, 'B_hit', 4, '회기동 B_hit(4)'), ('guD1', Gn1, 'R1', 0, '동대문구 R1'), ('guD1', Gn1, 'B_hit', 4, '동대문구 B_hit(4)')]):
    L = LAB[(sc, c, p, 8, 0)]; zdf = pd.DataFrame(Xn_, columns=cols).groupby(L).mean()
    im = ax.imshow(zdf.values, cmap='RdBu_r', vmin=-2, vmax=2, aspect='auto')
    ax.set_xticks(range(5)); ax.set_xticklabels(['용적률', '건폐율', '높이', '총층수', '나이'])
    ax.set_yticks(range(len(zdf))); ax.set_yticklabels([f'{i} (n={int((L == i).sum())})' for i in zdf.index]); ax.set_title(t)
    for (i, j), v in np.ndenumerate(zdf.values): ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=8)
cax = fig.add_axes([0.92, 0.2, 0.012, 0.6]); fig.colorbar(im, cax=cax, label='D1 안에서의 z-score 평균'); fig.suptitle('D1 k=8 시드 0 군집 프로파일')
fig.savefig('/content/figs_f2/g07_D1_profiles_heatmap.png', dpi=120); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4)); x = np.arange(len(KS))
ax.bar(x - .2, [F2['f2_3']['S_D0'][k] for k in KS], .4, label='D0 (0 값 포함)', color='tab:gray')
ax.bar(x + .2, [F2['f2_3']['S_D1'][k] for k in KS], .4, label='D1 (0 값 제외)', color='tab:blue')
ax.set_xticks(x); ax.set_xticklabels([f'k={k}' for k in KS]); ax.set_ylabel('R1 공간 응집 S (시드 중앙값)')
ax.set_title(f"회기동 판정 f2-3: {F2['f2_3']['verdict']} (기준: 0.05 이상 감소가 3개 k 이상)"); ax.legend(); fig.tight_layout()
fig.savefig('/content/figs_f2/g08_S_D0_vs_D1.png', dpi=120); plt.close(fig)

m, bmu = SOMS[('hoegiD1', 4, 0, (20, 20))]; h2 = np.bincount(bmu, minlength=400)
nl = node_labels_hit(m.map_ngf, h2, 8, 0).astype(float); nl[h2 == 0] = np.nan
fig, axs = plt.subplots(1, 2, figsize=(11, 4.6))
im = axs[0].imshow(np.where(h2 > 0, h2, np.nan).reshape(20, 20), origin='lower', cmap='viridis'); fig.colorbar(im, ax=axs[0], shrink=.8)
axs[0].set_title(f'회기동 D1 tol=4 시드 0: 노드별 건물 수 (흰 칸 = 0, {int((h2 == 0).sum())}개)', fontsize=10)
axs[1].imshow(nl.reshape(20, 20), origin='lower', cmap='tab10', vmin=-.5, vmax=9.5); axs[1].set_title('B_hit(4) k=8 노드 군집 (흰 칸 = 묶지 않은 빈 노드)', fontsize=10)
for a in axs: a.set_xticks([]); a.set_yticks([])
fig.tight_layout(); fig.savefig('/content/figs_f2/g09_node_hits_D1.png', dpi=120); plt.close(fig)

if HAS_ZONING:  # g10 외부 변수 (용도지역 파일이 있을 때만)
    items = [('용도지역\n회기동', 'zoning', 'hoegi'), ('용도지역\n동대문구', 'zoning', 'gu'), ('주용도\n회기동', 'use', 'hoegi'), ('주용도\n동대문구', 'use', 'gu'), ('법정동\n동대문구', 'dong', 'gu')]
    def d0v(c, var, sc):
        cc = 'B' if c == 'B_hit' else c
        if var == 'zoning': return S4['zoning']['V'][cc][sc]
        if var == 'use': return S4['V_use'][cc][sc]
        return S4['V_dong_gu'][cc]
    fig, axs = plt.subplots(1, 2, figsize=(17, 5), sharey=True); x = np.arange(len(items)); wdt = .26
    for ax, ver in zip(axs, ['D0', 'D1']):
        for j, (c, col, lab) in enumerate([('R1', 'tab:blue', 'R1 속성만'), ('R2', 'tab:gray', 'R2 좌표만'), ('B_hit', 'tab:orange', 'B(4) (D0는 B_all, D1은 B_hit)')]):
            vals = [d0v(c, v, sc) if ver == 'D0' else EXT[c][f'{v}_{sc}'] for _, v, sc in items]
            ax.bar(x + (j - 1) * wdt, vals, wdt, color=col, label=lab)
            for xi, vv in zip(x + (j - 1) * wdt, vals): ax.text(xi, vv + .01, f'{vv:.2f}', ha='center', fontsize=7)
        ax.axhline(.3, ls='--', c='k', lw=.8); ax.set_xticks(x); ax.set_xticklabels([t for t, _, _ in items]); ax.set_title(f'{ver} (k=8)')
    axs[0].set_ylabel("Cramér's V"); axs[1].legend(fontsize=8); fig.suptitle('외부 변수와 군집의 관계: D0 vs D1 (점선 0.3 = 용도지역 재발견 읽는 규칙)')
    fig.tight_layout(); fig.savefig('/content/figs_f2/g10_external_D1.png', dpi=120); plt.close(fig)

fig, axs = plt.subplots(1, 3, figsize=(18, 4.8))
pat = pd.DataFrame({'회기동': pd.Series(OBS['O1']['hoegi']), '동대문구': pd.Series(OBS['O1']['gu'])}).fillna(0)
pat = pat.div(pat.sum(0), axis=1).sort_values('동대문구', ascending=False)
pat.plot.bar(ax=axs[0], color=['tab:orange', 'tab:blue']); axs[0].set_title("0 조합 비율 (용적률·건폐율·높이 순, '0'=0 값, '.'=값 있음)", fontsize=10); axs[0].set_ylabel('비율')
lab2 = list(OBS['O2']['by_register'].keys()) + list(OBS['O2']['by_use_gu'].keys())
val2 = [OBS['O2']['by_register'][r]['anyzero'] for r in OBS['O2']['by_register']] + [OBS['O2']['by_use_gu'][u]['anyzero'] for u in OBS['O2']['by_use_gu']]
nn2 = [OBS['O2']['by_register'][r]['n'] for r in OBS['O2']['by_register']] + [OBS['O2']['by_use_gu'][u]['n'] for u in OBS['O2']['by_use_gu']]
axs[1].barh(range(len(lab2)), val2, color=['tab:red'] * len(OBS['O2']['by_register']) + ['tab:blue'] * len(OBS['O2']['by_use_gu']))
axs[1].set_yticks(range(len(lab2))); axs[1].set_yticklabels([f'{l} (n={n_})' for l, n_ in zip(lab2, nn2)], fontsize=8); axs[1].invert_yaxis()
axs[1].set_title('동대문구: 하나라도 0인 비율 (빨강 = 대장 구분, 파랑 = 주용도 상위 10)', fontsize=10)
jit = np.random.default_rng(0).uniform(-0.25, 0.25, mask_g.sum())
axs[2].scatter(Graw1['total_floor'] + jit, Graw1['A30'], s=2, alpha=.3, linewidths=0)
axs[2].set_xlim(0, 25); axs[2].set_ylim(0, 90); axs[2].set_xlabel('총층수 (지터)'); axs[2].set_ylabel('높이 (m)')
axs[2].set_title(f"동대문구 D1: 높이 vs 층수 r={OBS['O3']['corr_height_floor_D1_gu']:.2f}", fontsize=10)
fig.suptitle('결측 패턴 관찰 O1~O3'); fig.tight_layout(); fig.savefig('/content/figs_f2/g11_missing_patterns.png', dpi=120); plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4.5))
for sc, c, p, lab in [('hoegiD1', 'R1', 0, 'R1'), ('hoegiD1', 'R2', 0, 'R2'), ('hoegiD1', 'B_hit', 4, 'B_hit(4)'), ('hoegiD1e100', 'B_hit', 4, 'B_hit(4) epochs 100'),
                      ('hoegiD1', 'B_all', 4, 'B_all(4)'), ('hoegiD1', 'C_hit', 4, 'C_hit(4)')]:
    ax.plot(KS, [ari_within(sc, c, p, k) for k in KS], '-o', label=lab)
ax.axhline(.75, ls='--', c='k', lw=.7); ax.axhline(.5, ls=':', c='k', lw=.7)
ax.set_xlabel('k'); ax.set_ylabel('시드 쌍별 ARI 중앙값'); ax.set_title('회기동 D1 안정성 (선: 0.75 / 0.5)'); ax.legend(fontsize=8); fig.tight_layout()
fig.savefig('/content/figs_f2/g12_stability_D1.png', dpi=120); plt.close(fig)
print(sorted(os.listdir('/content/figs_f2')))

### 11-6. 결과 저장

In [ ]:
# 결과 요약 JSON (/content/results_analysis_f2.json)
F2_SCALES = ('hoegiD1', 'hoegiD1e100', 'guD1')
rows2 = [r for r in ROWS if r['scale'] in F2_SCALES or (r['scale'] == 'hoegi' and r['cand'] in ('B_hit', 'C_hit'))]
RESULTS_F2 = dict(meta=dict(N_D1_hoegi=int(mask_h.sum()), N_D1_gu=int(mask_g.sum()), KS=KS, SEEDS=SEEDS, WS=WS), OBS=OBS, F2=F2,
                  medians=pd.DataFrame(rows2).groupby(['scale', 'cand', 'param', 'k'])[['sil', 'r2', 'db', 'S', 'lift', 'k_eff']].median().round(4).reset_index().to_dict(orient='records'))
json.dump(RESULTS_F2, open('/content/results_analysis_f2.json', 'w'), ensure_ascii=False, default=float, indent=1)
print('저장: /content/results_analysis_f2.json')